# CruxSight.ai — Bottleneck Analysis Pipeline

**196-file structural bottleneck analysis on DeathStarBench**

Maral Alshanaa · 4th Year Graduation Project · 2026

## Project Overview

**Goal:** Analyze all bottleneck types and apply Theory of
Constraints to discover structural bottleneck patterns.
**Output:** Academic paper + GitHub repository + visualizations.

**Workflow per file:**
1. Download one file from the dataset
2. Analyze across Levels 2 → 3 → 4
3. Save results + charts to Google Drive
4. Delete the local file copy
5. Move to the next file

This produced 7 structural bottleneck patterns (A–G), with
Pattern D (Entry + Storage Core) dominating at 49.7% of
bottleneck files. The irreducible storage core
({13,14,20,21,26,27,28}) recurs across stress types —
establishing that constraints are architectural, not
resource-specific. See the main [README](../../README.md)
for full results.

**Reproducibility note:** the template cell below was manually
re-run with updated file parameters across multiple sessions
to process all 196 files (7 batches, varying batch sizes).
Each "Batch Results" markdown block documents the findings
from that batch before moving to the next.

# Microservices Bottleneck Localization — Analysis Notebook

**Goal:** Analyze all 196 processed files across 4 bottleneck types,
3 workflows, and 3 load levels. Apply Theory of Constraints to identify
structural constraints in the system.

**Final outputs:** Academic paper + GitHub + visualizations

**Analysis framework per file:**
- Level 2: System-level (class balance, flagged nodes, resource signals)
- Level 3: Signal analysis (latency ratios, correlations, detection power)
- Level 4: Modeling (RF, LR, GB comparison + ablation)

---

## Dataset structure
| Bottleneck type | Prefix | Files | Load levels |
|----------------|--------|-------|-------------|
| CPU stress | `cpu_` | ~60 | 200, 400, 800 RPS |
| Memory stress | `mem_` | ~40 | 800 RPS |
| CPU + Memory | `cpu_mem_` | ~40 | 800 RPS |
| Network throttle | `net_` | ~30 | 800 RPS |

## 1. Environment Setup
Mount Google Drive, install dependencies, restore session state.

In [ ]:
# ============================================================
# SETUP CELL — Run this once at the start of every session
# ============================================================
import os, zipfile, json, subprocess, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR   = '/content/drive/MyDrive/bottleneck_project'
RESULTS_DIR = f'{DRIVE_DIR}/results'
CHARTS_DIR  = f'{DRIVE_DIR}/charts'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHARTS_DIR,  exist_ok=True)

print("✓ All imports loaded")
print("✓ Drive mounted")
print("✓ Ready — now run Cell 5")

## 2. Dataset Download
Downloads the DeathStarBench dataset from Kaggle.
**Note:** requires your own `kaggle.json` credentials — not
included in this notebook for security. See the Kaggle API docs
to set up `~/.kaggle/kaggle.json` before running this cell.

In [ ]:
# Download the dataset
!kaggle datasets download -d gagansomashekar/microservices-bottleneck-detection-dataset

# Unzip
import zipfile, os

with zipfile.ZipFile('microservices-bottleneck-detection-dataset.zip', 'r') as z:
    z.extractall('microservices_data')
    print("Files extracted:")
    for f in z.namelist():
        print(" ", f)

# Preview each file
import pandas as pd

data_path = 'microservices_data'
files = [f for f in os.listdir(data_path) if f.endswith('.csv')]

dfs = {}
for f in files:
    df = pd.read_csv(os.path.join(data_path, f))
    dfs[f] = df
    print(f"\n{'='*50}")
    print(f"FILE: {f}")
    print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")
    print(df.head(3))

## 4. Reusable Analysis Template (Final Version)
The core pipeline, applied to each of the 196 files individually
by changing the 5 variables at the top of this cell:
`filename`, `bottleneck_type`, `workflow`, `rps`, `run_id`.

Performs a 4-level analysis (L2–L4), handles edge cases
(BASELINE_ONLY, BOTTLENECK_ONLY, TOO_SMALL files with <500
samples), generates charts, and saves JSON results + an English
markdown interpretation table to Drive.

**To reproduce the full 196-file analysis:** run this cell once
per file, updating the 5 variables each time. The aggregated
results across all runs produced the pattern taxonomy (A–G)
reported in the main paper.

**Reproducibility note:** the full 196-file analysis was executed
by manually updating the 5 variables above and re-running this
cell once per file across multiple sessions (not a single batch
loop). The aggregated JSON outputs from all 196 runs are available
in `results/` and form the basis of the pattern taxonomy (A–G)
reported in the thesis.

In [ ]:
# ============================================================
# CELL 5 — Corrected (fixed random_state typo)
# ============================================================

FILE_NAME   = 'cpu_july28_800_0_graph_1.csv'
BN_TYPE     = 'CPU stress'
WORKFLOW    = 'Compose'
LOAD_RPS    = 800
RUN_NUM     = 0

KAGGLE_PATH = (
    'processed_dataset/compose/multi-modal-data-separate/'
    + FILE_NAME
)
FILE_ID = (FILE_NAME
           .replace('_graph_1.csv','')
           .replace('_25min_repeat','')
           .replace('_25min_rerun','')
           .replace('_25min','')
           .replace('_30min',''))

LOCAL_DIR = '/content/ms_data'
local_csv = os.path.join(LOCAL_DIR, FILE_NAME)
os.makedirs(LOCAL_DIR, exist_ok=True)

if not os.path.exists(local_csv):
    print(f"Downloading {FILE_NAME}...")
    result = subprocess.run([
        'kaggle', 'datasets', 'download',
        'gagansomashekar/microservices-bottleneck-detection-dataset',
        '--path', LOCAL_DIR, '--file', KAGGLE_PATH
    ], capture_output=True, text=True)
    for f in os.listdir(LOCAL_DIR):
        if f.endswith('.zip'):
            with zipfile.ZipFile(os.path.join(LOCAL_DIR, f)) as z:
                z.extractall(LOCAL_DIR)
            os.remove(os.path.join(LOCAL_DIR, f))
    print("Downloaded and unzipped.")
else:
    print(f"Already exists: {FILE_NAME}")

df = pd.read_csv(local_csv)

latency_cols = [c for c in df.columns if c.endswith('_latency')]
rpc_cols     = [c for c in df.columns if c.endswith('_label_RPC')]
cpu_cols     = [c for c in df.columns if 'cpu' in c and 'label' not in c]
mem_cols     = [c for c in df.columns if 'memory' in c]
net_rx_cols  = [c for c in df.columns if 'receive' in c]
net_tx_cols  = [c for c in df.columns if 'transmit' in c]

n_total      = len(df)
n_normal     = int((df['label_trace']==0).sum())
n_bottleneck = int((df['label_trace']==1).sum())
bn_rate      = n_bottleneck / n_total
normal_df    = df[df['label_trace']==0]
bottleneck_df= df[df['label_trace']==1]
y            = df['label_trace'].values

print(f"\nLoaded: {FILE_NAME}")
print(f"  {n_total:,} traces | Normal: {n_normal:,} | "
      f"BN: {n_bottleneck:,} | Rate: {bn_rate*100:.1f}%")

MIN_TRACES   = 500
BOTH_CLASSES = (n_normal > 0 and n_bottleneck > 0)
ENOUGH_DATA  = (n_total >= MIN_TRACES)

if not BOTH_CLASSES:
    status = 'BASELINE_ONLY' if n_bottleneck == 0 else 'BOTTLENECK_ONLY'
elif not ENOUGH_DATA:
    status = 'TOO_SMALL'
else:
    status = 'OK'

if status != 'OK':
    print(f"\n⚠ Skipping modeling — status: {status}")
    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    n_lat = normal_df['0_latency']/1000 if n_normal > 0 else None
    b_lat = bottleneck_df['0_latency']/1000 if n_bottleneck > 0 else None
    print(f"  Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    if n_lat is not None:
        print(f"  Normal p50: {n_lat.median():.1f}ms")
    if b_lat is not None:
        print(f"  BN p50: {b_lat.median():.1f}ms")
    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4),
        'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  Saved: {results_path}")
    os.remove(local_csv)
    print(f"  Local file deleted.")
    print(f"\n{'='*50}")
    print(f"⚠ {FILE_ID} — {status}")
    print(f"{'='*50}")

else:
    n_lat = normal_df['0_latency'] / 1000
    b_lat = bottleneck_df['0_latency'] / 1000

    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    never_set   = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates==0].index)
    df['n_bn_rpcs'] = df[rpc_cols].sum(axis=1)
    multi = df[df['label_trace']==1]['n_bn_rpcs']

    res_ratios = {
        'cpu':    bottleneck_df[cpu_cols].mean().mean() /
                  normal_df[cpu_cols].mean().mean(),
        'memory': bottleneck_df[mem_cols].mean().mean() /
                  normal_df[mem_cols].mean().mean(),
        'net_rx': bottleneck_df[net_rx_cols].mean().mean() /
                  normal_df[net_rx_cols].mean().mean(),
        'net_tx': bottleneck_df[net_tx_cols].mean().mean() /
                  normal_df[net_tx_cols].mean().mean(),
    }

    print(f"\n[L2] Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    print(f"     Never flagged ({len(never_set)}): "
          f"{sorted(never_set, key=int)}")
    print(f"     Simultaneous RPCs: always {int(multi.mode()[0])} "
          f"| Rigid: {multi.nunique()==1}")
    print(f"     Entry p50 — Normal: {n_lat.median():.1f}ms  "
          f"BN: {b_lat.median():.1f}ms  "
          f"Ratio: {b_lat.median()/n_lat.median():.2f}x")
    print(f"     Resources: CPU={res_ratios['cpu']:.4f}x  "
          f"Mem={res_ratios['memory']:.4f}x  "
          f"NetRX={res_ratios['net_rx']:.4f}x")

    node_ratios = {}
    for col in latency_cols:
        node = col.replace('_latency','')
        nv   = normal_df[col].mean() / 1000
        bv   = bottleneck_df[col].mean() / 1000
        node_ratios[node] = {
            'normal_ms': round(nv,3), 'bn_ms': round(bv,3),
            'abs_diff':  round(bv-nv,3),
            'ratio':     round(bv/nv,4) if nv>0 else 0,
            'flagged':   node in flagged_set
        }

    lat_df = df[latency_cols].copy()
    lat_df.columns = [c.replace('_latency','') for c in latency_cols]
    corr   = lat_df.corr()
    fl     = [c for c in lat_df.columns if c in flagged_set]
    unfl   = [c for c in lat_df.columns if c not in flagged_set]
    ff     = corr.loc[fl, fl].values.copy() if len(fl)>1 else np.array([[np.nan]])
    np.fill_diagonal(ff, np.nan)
    fu     = corr.loc[fl, unfl].values if (len(fl)>0 and len(unfl)>0) else np.array([[0]])

    X_lat = df[latency_cols].values
    X_res = df[cpu_cols + mem_cols + net_rx_cols + net_tx_cols].values

    feature_aucs = {}
    for name, X in [('Latency only', X_lat),
                    ('Resource only', X_res)]:
        sc = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            X, y, cv=5, scoring='roc_auc')
        feature_aucs[name] = {'mean': round(sc.mean(),4),
                              'std':  round(sc.std(),4)}

    print(f"\n[L3] Latency-only AUC: "
          f"{feature_aucs['Latency only']['mean']:.4f}  "
          f"Resource-only AUC: "
          f"{feature_aucs['Resource only']['mean']:.4f}")
    print(f"     FF corr: {np.nanmean(ff):.4f}  "
          f"FU corr: {np.nanmean(fu):.4f}")

    scaler   = StandardScaler()
    X_lat_sc = scaler.fit_transform(X_lat)
    models   = {
        'Logistic Regression': (LogisticRegression(max_iter=1000,
                                 random_state=42), X_lat_sc),
        'Random Forest':       (RandomForestClassifier(n_estimators=200,
                                 random_state=42, n_jobs=-1), X_lat),
        'Gradient Boosting':   (GradientBoostingClassifier(n_estimators=100,
                                 random_state=42), X_lat),  # fixed
    }
    model_results = {}
    for name, (model, X) in models.items():
        auc_s = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
        f1_s  = cross_val_score(model, X, y, cv=5, scoring='f1')
        model_results[name] = {'auc': round(auc_s.mean(),4),
                               'f1':  round(f1_s.mean(),4),
                               'std': round(auc_s.std(),4)}

    baseline_auc = cross_val_score(
        RandomForestClassifier(n_estimators=100,
                               random_state=42, n_jobs=-1),
        X_lat, y, cv=5, scoring='roc_auc').mean()

    ablation = {}
    for i, col in enumerate(latency_cols):
        node = col.replace('_latency','')
        sc   = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            np.delete(X_lat, i, axis=1), y, cv=3,
            scoring='roc_auc').mean()
        ablation[node] = round(sc,4)

    drops   = {n: round(baseline_auc-sc,4) for n,sc in ablation.items()}
    drops_s = sorted(drops.items(), key=lambda x: x[1], reverse=True)

    print(f"\n[L4] RF AUC: {model_results['Random Forest']['auc']:.4f}  "
          f"F1: {model_results['Random Forest']['f1']:.4f}")
    print(f"     LR AUC: "
          f"{model_results['Logistic Regression']['auc']:.4f}  "
          f"GB AUC: {model_results['Gradient Boosting']['auc']:.4f}")
    print(f"     Top ablation: {drops_s[:3]}")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Analysis: {FILE_NAME}\n'
                 f'{WORKFLOW} | {BN_TYPE} | {LOAD_RPS} RPS | Run {RUN_NUM}',
                 fontsize=12, fontweight='bold')

    ax = axes[0,0]
    ax.bar(['Normal','Bottleneck'], [n_normal, n_bottleneck],
           color=['#2196F3','#F44336'], edgecolor='white', width=0.5)
    ax.set_title('Class balance', fontweight='bold')
    ax.set_ylim(0, max(n_normal, n_bottleneck)*1.25)
    for i, val in enumerate([n_normal, n_bottleneck]):
        ax.text(i, val + max(n_normal,n_bottleneck)*0.02,
                f'{val:,}\n({val/n_total*100:.1f}%)',
                ha='center', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[0,1]
    ax.bar(range(30), [1]*30,
           color=['#F44336' if str(n) in flagged_set
                  else '#E0E0E0' for n in range(30)],
           edgecolor='white', width=0.8)
    ax.set_xticks(range(30))
    ax.set_xticklabels([str(n) for n in range(30)], fontsize=6, rotation=90)
    ax.set_yticks([])
    ax.set_title(f'Flagged nodes ({len(flagged_set)}/30)', fontweight='bold')
    ax.spines[['top','right','left']].set_visible(False)

    ax = axes[0,2]
    nodes_s  = sorted(node_ratios.keys(), key=int)
    ratios_v = [node_ratios[n]['ratio'] for n in nodes_s]
    ax.bar(range(30), ratios_v,
           color=['#F44336' if node_ratios[n]['flagged']
                  else '#90CAF9' for n in nodes_s],
           edgecolor='white', width=0.7)
    ax.axhline(1.0, color='gray', linestyle='--', lw=1, alpha=0.7)
    ax.set_xticks(range(30))
    ax.set_xticklabels(nodes_s, fontsize=6, rotation=90)
    ax.set_title('Per-node latency ratio', fontweight='bold')
    ax.set_ylabel('BN / Normal')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,0]
    rv = list(res_ratios.values())
    ax.barh(list(res_ratios.keys()), rv,
            color=['#EF5350' if v>1.1 else '#FFA726'
                   if v>1.03 else '#66BB6A' for v in rv],
            edgecolor='white')
    ax.axvline(1.0, color='gray', linestyle='--', lw=1)
    ax.set_title('Resource ratios', fontweight='bold')
    for i, v in enumerate(rv):
        ax.text(v+0.001, i, f'{v:.4f}x', va='center', fontsize=9)
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,1]
    fa_vals = [feature_aucs[k]['mean'] for k in feature_aucs]
    ax.bar(range(2), fa_vals, color=['#4CAF50','#FF9800'],
           edgecolor='white', width=0.4)
    ax.set_xticks(range(2))
    ax.set_xticklabels(['Latency\nonly','Resource\nonly'], fontsize=9)
    ax.set_ylim(0.5, 1.05)
    ax.set_title('AUC by feature set (RF)', fontweight='bold')
    for i, v in enumerate(fa_vals):
        ax.text(i, v+0.008, f'{v:.4f}', ha='center',
                fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,2]
    ax.barh(range(8), [v for _,v in drops_s[:8]][::-1],
            color=['#D32F2F' if v>0.008 else '#FF9800'
                   if v>0.004 else '#FFC107'
                   for _,v in drops_s[:8]][::-1],
            edgecolor='white')
    ax.set_yticks(range(8))
    ax.set_yticklabels([f'node {n}' for n,_ in drops_s[:8]][::-1], fontsize=8)
    ax.set_title(f'Node ablation (base={baseline_auc:.4f})', fontweight='bold')
    ax.set_xlabel('AUC drop')
    ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    chart_path = os.path.join(CHARTS_DIR, f'{FILE_ID}_analysis.png')
    plt.savefig(chart_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Chart saved: {chart_path}")

    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4), 'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
        'never_flagged': sorted(never_set, key=int),
        'n_flagged': len(flagged_set),
        'always_same_subgraph': bool(multi.nunique()==1),
        'simultaneous_rpcs': int(multi.mode()[0]),
        'entry_latency': {
            'normal_p50_ms': round(n_lat.median(),2),
            'bn_p50_ms':     round(b_lat.median(),2),
            'ratio_p50':     round(b_lat.median()/n_lat.median(),4),
            'ratio_mean':    round(b_lat.mean()/n_lat.mean(),4),
        },
        'resource_ratios': {k: round(v,4) for k,v in res_ratios.items()},
        'feature_aucs':   feature_aucs,
        'model_results':  model_results,
        'baseline_auc':   round(baseline_auc,4),
        'ablation_top8':  {n: v for n,v in drops_s[:8]},
        'node_ratios':    node_ratios,
        'ff_corr': round(float(np.nanmean(ff)),4),
        'fu_corr': round(float(np.nanmean(fu)),4),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Results saved: {results_path}")
    os.remove(local_csv)
    print(f"Local file deleted.")
    print(f"\n{'='*50}")
    print(f"✓ {FILE_ID} COMPLETE")
    print(f"{'='*50}")

## Batch 1 Results: CPU bottleneck — 200 RPS — Runs 0 to 9
*Files: cpu_aug12_25min_200_[0-9]_graph_1.csv | 10 files | ~148,000 traces*

### Key findings

**Subgraph is perfectly rigid across all 10 runs:**
Every bottleneck trace flags exactly the same 15 nodes simultaneously —
nodes 4, 5, 7, 8, 11, 12, 13, 14, 18, 19, 20, 21, 26, 27, 28.
Zero variation across 10 independent runs. The CPU bottleneck at 200 RPS
creates a deterministic constraint pattern.

**Quantitative stability:**
| Metric | Min | Max | Interpretation |
|--------|-----|-----|----------------|
| BN rate | 39.7% | 42.3% | Stable by design |
| Entry latency ratio (p50) | 1.31x | 1.36x | Weak surface signal |
| RF AUC (latency only) | 0.975 | 0.981 | Highly stable detection |
| Resource CPU ratio | 1.00x | 1.16x | Noisy, unreliable |

**Node 14 is consistently the single most important node** —
top ablation rank in all 10 runs (AUC drop 0.006–0.009).
Node 5 takes second place in 8 of 10 runs.

**Resource signals are unreliable:** CPU ratio ranges from 1.00x to 1.16x
across runs with identical experimental conditions. Latency is the only
stable detection signal.

### Theory of Constraints implication
The 15-node subgraph is the system constraint for CPU-stressed compose
at 200 RPS. It is not transient — 10 runs confirm it is structurally fixed.
This is the first anchor point for ToC application.

### Open question for next batch
Does the flagged node set change at 400 RPS?
From 800 RPS analysis we know it shrinks to 9 nodes.
400 RPS will reveal whether the transition is gradual or sudden.

---
## Batch 2: CPU bottleneck — 400 RPS — Runs 0 to 9
*Files: cpu_aug9_25min_400_[0-9]_graph_1.csv*
*Question: How does the constraint subgraph change with higher load?*

In [ ]:
# ============================================================
# CELL 5 — Corrected (re-run for batch 2)
# ============================================================

FILE_NAME   = 'cpu_july28_800_0_graph_1.csv'
BN_TYPE     = 'CPU stress'
WORKFLOW    = 'Compose'
LOAD_RPS    = 800
RUN_NUM     = 0

KAGGLE_PATH = (
    'processed_dataset/compose/multi-modal-data-separate/'
    + FILE_NAME
)
FILE_ID = (FILE_NAME
           .replace('_graph_1.csv','')
           .replace('_25min_repeat','')
           .replace('_25min_rerun','')
           .replace('_25min','')
           .replace('_30min',''))

LOCAL_DIR = '/content/ms_data'
local_csv = os.path.join(LOCAL_DIR, FILE_NAME)
os.makedirs(LOCAL_DIR, exist_ok=True)

if not os.path.exists(local_csv):
    print(f"Downloading {FILE_NAME}...")
    result = subprocess.run([
        'kaggle', 'datasets', 'download',
        'gagansomashekar/microservices-bottleneck-detection-dataset',
        '--path', LOCAL_DIR, '--file', KAGGLE_PATH
    ], capture_output=True, text=True)
    for f in os.listdir(LOCAL_DIR):
        if f.endswith('.zip'):
            with zipfile.ZipFile(os.path.join(LOCAL_DIR, f)) as z:
                z.extractall(LOCAL_DIR)
            os.remove(os.path.join(LOCAL_DIR, f))
    print("Downloaded and unzipped.")
else:
    print(f"Already exists: {FILE_NAME}")

df = pd.read_csv(local_csv)

latency_cols = [c for c in df.columns if c.endswith('_latency')]
rpc_cols     = [c for c in df.columns if c.endswith('_label_RPC')]
cpu_cols     = [c for c in df.columns if 'cpu' in c and 'label' not in c]
mem_cols     = [c for c in df.columns if 'memory' in c]
net_rx_cols  = [c for c in df.columns if 'receive' in c]
net_tx_cols  = [c for c in df.columns if 'transmit' in c]

n_total      = len(df)
n_normal     = int((df['label_trace']==0).sum())
n_bottleneck = int((df['label_trace']==1).sum())
bn_rate      = n_bottleneck / n_total
normal_df    = df[df['label_trace']==0]
bottleneck_df= df[df['label_trace']==1]
y            = df['label_trace'].values

print(f"\nLoaded: {FILE_NAME}")
print(f"  {n_total:,} traces | Normal: {n_normal:,} | "
      f"BN: {n_bottleneck:,} | Rate: {bn_rate*100:.1f}%")

MIN_TRACES   = 500
BOTH_CLASSES = (n_normal > 0 and n_bottleneck > 0)
ENOUGH_DATA  = (n_total >= MIN_TRACES)

if not BOTH_CLASSES:
    status = 'BASELINE_ONLY' if n_bottleneck == 0 else 'BOTTLENECK_ONLY'
elif not ENOUGH_DATA:
    status = 'TOO_SMALL'
else:
    status = 'OK'

if status != 'OK':
    print(f"\n⚠ Skipping modeling — status: {status}")
    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    n_lat = normal_df['0_latency']/1000 if n_normal > 0 else None
    b_lat = bottleneck_df['0_latency']/1000 if n_bottleneck > 0 else None
    print(f"  Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    if n_lat is not None:
        print(f"  Normal p50: {n_lat.median():.1f}ms")
    if b_lat is not None:
        print(f"  BN p50: {b_lat.median():.1f}ms")
    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4),
        'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  Saved: {results_path}")
    os.remove(local_csv)
    print(f"  Local file deleted.")
    print(f"\n{'='*50}")
    print(f"⚠ {FILE_ID} — {status}")
    print(f"{'='*50}")

else:
    n_lat = normal_df['0_latency'] / 1000
    b_lat = bottleneck_df['0_latency'] / 1000

    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    never_set   = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates==0].index)
    df['n_bn_rpcs'] = df[rpc_cols].sum(axis=1)
    multi = df[df['label_trace']==1]['n_bn_rpcs']

    res_ratios = {
        'cpu':    bottleneck_df[cpu_cols].mean().mean() /
                  normal_df[cpu_cols].mean().mean(),
        'memory': bottleneck_df[mem_cols].mean().mean() /
                  normal_df[mem_cols].mean().mean(),
        'net_rx': bottleneck_df[net_rx_cols].mean().mean() /
                  normal_df[net_rx_cols].mean().mean(),
        'net_tx': bottleneck_df[net_tx_cols].mean().mean() /
                  normal_df[net_tx_cols].mean().mean(),
    }

    print(f"\n[L2] Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    print(f"     Never flagged ({len(never_set)}): "
          f"{sorted(never_set, key=int)}")
    print(f"     Simultaneous RPCs: always {int(multi.mode()[0])} "
          f"| Rigid: {multi.nunique()==1}")
    print(f"     Entry p50 — Normal: {n_lat.median():.1f}ms  "
          f"BN: {b_lat.median():.1f}ms  "
          f"Ratio: {b_lat.median()/n_lat.median():.2f}x")
    print(f"     Resources: CPU={res_ratios['cpu']:.4f}x  "
          f"Mem={res_ratios['memory']:.4f}x  "
          f"NetRX={res_ratios['net_rx']:.4f}x")

    node_ratios = {}
    for col in latency_cols:
        node = col.replace('_latency','')
        nv   = normal_df[col].mean() / 1000
        bv   = bottleneck_df[col].mean() / 1000
        node_ratios[node] = {
            'normal_ms': round(nv,3), 'bn_ms': round(bv,3),
            'abs_diff':  round(bv-nv,3),
            'ratio':     round(bv/nv,4) if nv>0 else 0,
            'flagged':   node in flagged_set
        }

    lat_df = df[latency_cols].copy()
    lat_df.columns = [c.replace('_latency','') for c in latency_cols]
    corr   = lat_df.corr()
    fl     = [c for c in lat_df.columns if c in flagged_set]
    unfl   = [c for c in lat_df.columns if c not in flagged_set]
    ff     = corr.loc[fl, fl].values.copy() if len(fl)>1 else np.array([[np.nan]])
    np.fill_diagonal(ff, np.nan)
    fu     = corr.loc[fl, unfl].values if (len(fl)>0 and len(unfl)>0) else np.array([[0]])

    X_lat = df[latency_cols].values
    X_res = df[cpu_cols + mem_cols + net_rx_cols + net_tx_cols].values

    feature_aucs = {}
    for name, X in [('Latency only', X_lat),
                    ('Resource only', X_res)]:
        sc = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            X, y, cv=5, scoring='roc_auc')
        feature_aucs[name] = {'mean': round(sc.mean(),4),
                              'std':  round(sc.std(),4)}

    print(f"\n[L3] Latency-only AUC: "
          f"{feature_aucs['Latency only']['mean']:.4f}  "
          f"Resource-only AUC: "
          f"{feature_aucs['Resource only']['mean']:.4f}")
    print(f"     FF corr: {np.nanmean(ff):.4f}  "
          f"FU corr: {np.nanmean(fu):.4f}")

    scaler   = StandardScaler()
    X_lat_sc = scaler.fit_transform(X_lat)
    models   = {
        'Logistic Regression': (LogisticRegression(max_iter=1000,
                                 random_state=42), X_lat_sc),
        'Random Forest':       (RandomForestClassifier(n_estimators=200,
                                 random_state=42, n_jobs=-1), X_lat),
        'Gradient Boosting':   (GradientBoostingClassifier(n_estimators=100,
                                 random_state=42), X_lat),  # fixed
    }
    model_results = {}
    for name, (model, X) in models.items():
        auc_s = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
        f1_s  = cross_val_score(model, X, y, cv=5, scoring='f1')
        model_results[name] = {'auc': round(auc_s.mean(),4),
                               'f1':  round(f1_s.mean(),4),
                               'std': round(auc_s.std(),4)}

    baseline_auc = cross_val_score(
        RandomForestClassifier(n_estimators=100,
                               random_state=42, n_jobs=-1),
        X_lat, y, cv=5, scoring='roc_auc').mean()

    ablation = {}
    for i, col in enumerate(latency_cols):
        node = col.replace('_latency','')
        sc   = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            np.delete(X_lat, i, axis=1), y, cv=3,
            scoring='roc_auc').mean()
        ablation[node] = round(sc,4)

    drops   = {n: round(baseline_auc-sc,4) for n,sc in ablation.items()}
    drops_s = sorted(drops.items(), key=lambda x: x[1], reverse=True)

    print(f"\n[L4] RF AUC: {model_results['Random Forest']['auc']:.4f}  "
          f"F1: {model_results['Random Forest']['f1']:.4f}")
    print(f"     LR AUC: "
          f"{model_results['Logistic Regression']['auc']:.4f}  "
          f"GB AUC: {model_results['Gradient Boosting']['auc']:.4f}")
    print(f"     Top ablation: {drops_s[:3]}")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Analysis: {FILE_NAME}\n'
                 f'{WORKFLOW} | {BN_TYPE} | {LOAD_RPS} RPS | Run {RUN_NUM}',
                 fontsize=12, fontweight='bold')

    ax = axes[0,0]
    ax.bar(['Normal','Bottleneck'], [n_normal, n_bottleneck],
           color=['#2196F3','#F44336'], edgecolor='white', width=0.5)
    ax.set_title('Class balance', fontweight='bold')
    ax.set_ylim(0, max(n_normal, n_bottleneck)*1.25)
    for i, val in enumerate([n_normal, n_bottleneck]):
        ax.text(i, val + max(n_normal,n_bottleneck)*0.02,
                f'{val:,}\n({val/n_total*100:.1f}%)',
                ha='center', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[0,1]
    ax.bar(range(30), [1]*30,
           color=['#F44336' if str(n) in flagged_set
                  else '#E0E0E0' for n in range(30)],
           edgecolor='white', width=0.8)
    ax.set_xticks(range(30))
    ax.set_xticklabels([str(n) for n in range(30)], fontsize=6, rotation=90)
    ax.set_yticks([])
    ax.set_title(f'Flagged nodes ({len(flagged_set)}/30)', fontweight='bold')
    ax.spines[['top','right','left']].set_visible(False)

    ax = axes[0,2]
    nodes_s  = sorted(node_ratios.keys(), key=int)
    ratios_v = [node_ratios[n]['ratio'] for n in nodes_s]
    ax.bar(range(30), ratios_v,
           color=['#F44336' if node_ratios[n]['flagged']
                  else '#90CAF9' for n in nodes_s],
           edgecolor='white', width=0.7)
    ax.axhline(1.0, color='gray', linestyle='--', lw=1, alpha=0.7)
    ax.set_xticks(range(30))
    ax.set_xticklabels(nodes_s, fontsize=6, rotation=90)
    ax.set_title('Per-node latency ratio', fontweight='bold')
    ax.set_ylabel('BN / Normal')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,0]
    rv = list(res_ratios.values())
    ax.barh(list(res_ratios.keys()), rv,
            color=['#EF5350' if v>1.1 else '#FFA726'
                   if v>1.03 else '#66BB6A' for v in rv],
            edgecolor='white')
    ax.axvline(1.0, color='gray', linestyle='--', lw=1)
    ax.set_title('Resource ratios', fontweight='bold')
    for i, v in enumerate(rv):
        ax.text(v+0.001, i, f'{v:.4f}x', va='center', fontsize=9)
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,1]
    fa_vals = [feature_aucs[k]['mean'] for k in feature_aucs]
    ax.bar(range(2), fa_vals, color=['#4CAF50','#FF9800'],
           edgecolor='white', width=0.4)
    ax.set_xticks(range(2))
    ax.set_xticklabels(['Latency\nonly','Resource\nonly'], fontsize=9)
    ax.set_ylim(0.5, 1.05)
    ax.set_title('AUC by feature set (RF)', fontweight='bold')
    for i, v in enumerate(fa_vals):
        ax.text(i, v+0.008, f'{v:.4f}', ha='center',
                fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,2]
    ax.barh(range(8), [v for _,v in drops_s[:8]][::-1],
            color=['#D32F2F' if v>0.008 else '#FF9800'
                   if v>0.004 else '#FFC107'
                   for _,v in drops_s[:8]][::-1],
            edgecolor='white')
    ax.set_yticks(range(8))
    ax.set_yticklabels([f'node {n}' for n,_ in drops_s[:8]][::-1], fontsize=8)
    ax.set_title(f'Node ablation (base={baseline_auc:.4f})', fontweight='bold')
    ax.set_xlabel('AUC drop')
    ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    chart_path = os.path.join(CHARTS_DIR, f'{FILE_ID}_analysis.png')
    plt.savefig(chart_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Chart saved: {chart_path}")

    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4), 'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
        'never_flagged': sorted(never_set, key=int),
        'n_flagged': len(flagged_set),
        'always_same_subgraph': bool(multi.nunique()==1),
        'simultaneous_rpcs': int(multi.mode()[0]),
        'entry_latency': {
            'normal_p50_ms': round(n_lat.median(),2),
            'bn_p50_ms':     round(b_lat.median(),2),
            'ratio_p50':     round(b_lat.median()/n_lat.median(),4),
            'ratio_mean':    round(b_lat.mean()/n_lat.mean(),4),
        },
        'resource_ratios': {k: round(v,4) for k,v in res_ratios.items()},
        'feature_aucs':   feature_aucs,
        'model_results':  model_results,
        'baseline_auc':   round(baseline_auc,4),
        'ablation_top8':  {n: v for n,v in drops_s[:8]},
        'node_ratios':    node_ratios,
        'ff_corr': round(float(np.nanmean(ff)),4),
        'fu_corr': round(float(np.nanmean(fu)),4),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Results saved: {results_path}")
    os.remove(local_csv)
    print(f"Local file deleted.")
    print(f"\n{'='*50}")
    print(f"✓ {FILE_ID} COMPLETE")
    print(f"{'='*50}")

## Batch 2 Results: CPU bottleneck — 400 RPS — Runs 0 to 9
*Files: cpu_aug9_25min_400_[0-9]_graph_1.csv | 10 files | ~280,000 traces*

### Important discovery: runs 0, 1, 2 are BASELINE-ONLY files
Runs 0, 1, and 2 contain zero bottleneck traces — no injection occurred.
This is scientifically valuable: they give us the **clean 400 RPS baseline**
latency distribution without any stress applied.

**400 RPS baseline (node 0, no bottleneck):**
| Metric | Run 0 | Run 1 | Run 2 |
|--------|-------|-------|-------|
| p50 | 17.4ms | 17.5ms | 17.3ms |
| p95 | 34.1ms | 34.3ms | 33.9ms |
| p99 | 45.5ms | 46.3ms | 45.7ms |

The baseline is remarkably stable across runs and nearly identical
to 200 RPS (p50=17.8ms). This tells us 400 RPS does not significantly
increase baseline latency — the system is not yet saturated at this load.

### The flagged subgraph does NOT change at 400 RPS
All 7 valid runs (3–9) flag **exactly the same 15 nodes** as 200 RPS:
nodes 4, 5, 7, 8, 11, 12, 13, 14, 18, 19, 20, 21, 26, 27, 28.
The constraint subgraph is identical at both 200 and 400 RPS.
The transition to 9 nodes only happens at 800 RPS.

### Quantitative comparison: 200 RPS vs 400 RPS
| Metric | 200 RPS (runs 0-9) | 400 RPS (runs 3-9) |
|--------|-------------------|-------------------|
| Flagged nodes | 15 (always) | 15 (always) |
| Entry p50 ratio | 1.31–1.36x | 1.35–1.47x |
| RF AUC | 0.975–0.981 | 0.955–0.974 |
| LR AUC | 0.803–0.832 | 0.813–0.847 |
| Resource CPU ratio | 1.00–1.16x | 0.81–1.31x |

**AUC is slightly lower at 400 RPS** (0.955–0.974 vs 0.975–0.981).
The signal is marginally harder to detect — but the subgraph is identical.

**Resource signals are MORE variable at 400 RPS** — ranging from 0.81x
(run 3, resources lower during bottleneck) to 1.31x (run 8). This
confirms resource metrics are unreliable for detection at any load level.

### Node 14 remains #1 across all 7 valid runs
Ablation top rank: node 14 in every single run (drop 0.007–0.009).
Node 5 consistently second. Node 4 consistently third.
This pattern is identical to 200 RPS — the same nodes drive detection
regardless of load level within the 15-node subgraph.

### Theory of Constraints implication
The constraint subgraph (15 nodes) is **load-invariant between 200 and
400 RPS**. The system constraint does not shift or expand when load
doubles from 200 to 400 RPS. The constraint only changes (shrinks to 9
nodes) when load reaches 800 RPS — suggesting a non-linear threshold
effect rather than a gradual transition.

### Open question for next batch (800 RPS)
We already know from earlier exploration that 800 RPS produces 9 flagged
nodes instead of 15. The question now is: **which 6 nodes disappear
from the subgraph at 800 RPS, and why?**
The 6 nodes that were flagged at 200/400 but not 800 RPS are:
7, 8, 11, 12, 18, 19. Understanding why these 6 become
undetectable at high load is a key finding for the paper.

---
## Batch 3: CPU bottleneck — 800 RPS
*Files: cpu_aug18, cpu_aug30, cpu_july*, cpu_sept* — multiple dates*
*Question: Which nodes disappear from the subgraph? Why?*

## Batch 3 Results: CPU bottleneck — 800 RPS — First 5 files
*Files: cpu_aug18 (runs 0-1) + cpu_aug30 (runs 0-2)*

### MAJOR FINDING: Two distinct subgraph patterns at 800 RPS

The 800 RPS files split into TWO completely different behaviors
depending on the recording date:

| Date | Flagged nodes | Count | Entry p50 ratio | RF AUC |
|------|--------------|-------|-----------------|--------|
| aug18 run 0 | 4,5,7,8,11,12,13,14,18,19,20,21,26,27,28 | **15** | 1.75x | 0.962 |
| aug18 run 1 | 4,5,7,8,11,12,13,14,18,19,20,21,26,27,28 | **15** | 1.72x | 0.968 |
| aug30 run 0 | 4,5,13,14,20,21,26,27,28 | **9** | 1.19x | 0.841 |
| aug30 run 1 | 4,5,13,14,20,21,26,27,28 | **9** | 1.19x | 0.815 |
| aug30 run 2 | 4,5,13,14,20,21,26,27,28 | **9** | 1.18x | 0.827 |

**aug18 at 800 RPS behaves like 200/400 RPS** — 15 nodes, strong signal
(ratio 1.72–1.75x), high AUC (0.962–0.968).

**aug30 at 800 RPS shows the compressed pattern** — 9 nodes, weak signal
(ratio 1.18–1.19x), significantly lower AUC (0.815–0.841).

### What explains the aug18 vs aug30 difference?

The experimental conditions differ between dates. Two hypotheses:

1. **Different injection intensity:** aug18 may inject heavier CPU stress,
   causing more widespread propagation through the call chain — activating
   all 15 nodes. aug30 may inject lighter stress, only affecting the 9
   core storage/timeline services.

2. **Different system baseline:** aug30 has higher normal baseline latency
   (p50=20–21ms) vs aug18 (p50=20–21ms) — similar, so this is less likely
   the cause.

3. **Different bottleneck target service:** The CPU stress may be injected
   on a different service between dates, causing a different propagation
   path through the dependency graph.

This is the most important finding so far — the subgraph is not just
load-dependent, it is also **injection-configuration-dependent**.

### Critical AUC degradation at aug30 800 RPS
aug30 RF AUC drops to 0.815–0.841 — a significant fall from 0.975–0.981
at 200 RPS. The 9-node pattern is genuinely harder to detect. The latency
ratio shrinks to only 1.18–1.19x at the entry point, meaning the 9
remaining flagged nodes produce barely visible surface signal.

### Ablation shift: node 4 overtakes node 14 at aug30
At 200/400 RPS: node 14 is consistently most important.
At aug30 800 RPS: **node 4 takes first place** with drops of 0.025–0.038.
Node 8 rises to second despite NOT being in the 9-node flagged set —
suggesting node 8 still carries indirect signal even when unlabeled.

### Resource signals at aug30 are anomalously high
aug30 run 0: CPU=1.64x, NetRX=1.64x — much higher than anything seen
before. Yet AUC is lower. This confirms that resource signals and
detection difficulty are decoupled — high resource ratios do not mean
easier detection.

### Theory of Constraints implication
There are at least TWO distinct CPU constraint patterns at 800 RPS:
- **Wide constraint (15 nodes):** aug18 — affects full compose chain
- **Core constraint (9 nodes):** aug30 — affects only storage/timeline core

The 9-node core {4,5,13,14,20,21,26,27,28} appears in BOTH patterns,
confirming it is the irreducible constraint — the true bottleneck.
The additional 6 nodes {7,8,11,12,18,19} are secondary effects that
appear only when stress is intense enough to propagate beyond the core.

---
## Continuing Batch 3: Remaining 800 RPS files
*cpu_aug30 runs 3-9, then cpu_july*, cpu_sept* files*
*Question: Which pattern (15-node or 9-node) dominates across all dates?*

In [ ]:
# ============================================================
# CELL 5 — Corrected (re-run for batch 3)
# ============================================================

FILE_NAME   = 'cpu_july28_800_0_graph_1.csv'
BN_TYPE     = 'CPU stress'
WORKFLOW    = 'Compose'
LOAD_RPS    = 800
RUN_NUM     = 0

KAGGLE_PATH = (
    'processed_dataset/compose/multi-modal-data-separate/'
    + FILE_NAME
)
FILE_ID = (FILE_NAME
           .replace('_graph_1.csv','')
           .replace('_25min_repeat','')
           .replace('_25min_rerun','')
           .replace('_25min','')
           .replace('_30min',''))

LOCAL_DIR = '/content/ms_data'
local_csv = os.path.join(LOCAL_DIR, FILE_NAME)
os.makedirs(LOCAL_DIR, exist_ok=True)

if not os.path.exists(local_csv):
    print(f"Downloading {FILE_NAME}...")
    result = subprocess.run([
        'kaggle', 'datasets', 'download',
        'gagansomashekar/microservices-bottleneck-detection-dataset',
        '--path', LOCAL_DIR, '--file', KAGGLE_PATH
    ], capture_output=True, text=True)
    for f in os.listdir(LOCAL_DIR):
        if f.endswith('.zip'):
            with zipfile.ZipFile(os.path.join(LOCAL_DIR, f)) as z:
                z.extractall(LOCAL_DIR)
            os.remove(os.path.join(LOCAL_DIR, f))
    print("Downloaded and unzipped.")
else:
    print(f"Already exists: {FILE_NAME}")

df = pd.read_csv(local_csv)

latency_cols = [c for c in df.columns if c.endswith('_latency')]
rpc_cols     = [c for c in df.columns if c.endswith('_label_RPC')]
cpu_cols     = [c for c in df.columns if 'cpu' in c and 'label' not in c]
mem_cols     = [c for c in df.columns if 'memory' in c]
net_rx_cols  = [c for c in df.columns if 'receive' in c]
net_tx_cols  = [c for c in df.columns if 'transmit' in c]

n_total      = len(df)
n_normal     = int((df['label_trace']==0).sum())
n_bottleneck = int((df['label_trace']==1).sum())
bn_rate      = n_bottleneck / n_total
normal_df    = df[df['label_trace']==0]
bottleneck_df= df[df['label_trace']==1]
y            = df['label_trace'].values

print(f"\nLoaded: {FILE_NAME}")
print(f"  {n_total:,} traces | Normal: {n_normal:,} | "
      f"BN: {n_bottleneck:,} | Rate: {bn_rate*100:.1f}%")

MIN_TRACES   = 500
BOTH_CLASSES = (n_normal > 0 and n_bottleneck > 0)
ENOUGH_DATA  = (n_total >= MIN_TRACES)

if not BOTH_CLASSES:
    status = 'BASELINE_ONLY' if n_bottleneck == 0 else 'BOTTLENECK_ONLY'
elif not ENOUGH_DATA:
    status = 'TOO_SMALL'
else:
    status = 'OK'

if status != 'OK':
    print(f"\n⚠ Skipping modeling — status: {status}")
    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    n_lat = normal_df['0_latency']/1000 if n_normal > 0 else None
    b_lat = bottleneck_df['0_latency']/1000 if n_bottleneck > 0 else None
    print(f"  Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    if n_lat is not None:
        print(f"  Normal p50: {n_lat.median():.1f}ms")
    if b_lat is not None:
        print(f"  BN p50: {b_lat.median():.1f}ms")
    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4),
        'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  Saved: {results_path}")
    os.remove(local_csv)
    print(f"  Local file deleted.")
    print(f"\n{'='*50}")
    print(f"⚠ {FILE_ID} — {status}")
    print(f"{'='*50}")

else:
    n_lat = normal_df['0_latency'] / 1000
    b_lat = bottleneck_df['0_latency'] / 1000

    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    never_set   = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates==0].index)
    df['n_bn_rpcs'] = df[rpc_cols].sum(axis=1)
    multi = df[df['label_trace']==1]['n_bn_rpcs']

    res_ratios = {
        'cpu':    bottleneck_df[cpu_cols].mean().mean() /
                  normal_df[cpu_cols].mean().mean(),
        'memory': bottleneck_df[mem_cols].mean().mean() /
                  normal_df[mem_cols].mean().mean(),
        'net_rx': bottleneck_df[net_rx_cols].mean().mean() /
                  normal_df[net_rx_cols].mean().mean(),
        'net_tx': bottleneck_df[net_tx_cols].mean().mean() /
                  normal_df[net_tx_cols].mean().mean(),
    }

    print(f"\n[L2] Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    print(f"     Never flagged ({len(never_set)}): "
          f"{sorted(never_set, key=int)}")
    print(f"     Simultaneous RPCs: always {int(multi.mode()[0])} "
          f"| Rigid: {multi.nunique()==1}")
    print(f"     Entry p50 — Normal: {n_lat.median():.1f}ms  "
          f"BN: {b_lat.median():.1f}ms  "
          f"Ratio: {b_lat.median()/n_lat.median():.2f}x")
    print(f"     Resources: CPU={res_ratios['cpu']:.4f}x  "
          f"Mem={res_ratios['memory']:.4f}x  "
          f"NetRX={res_ratios['net_rx']:.4f}x")

    node_ratios = {}
    for col in latency_cols:
        node = col.replace('_latency','')
        nv   = normal_df[col].mean() / 1000
        bv   = bottleneck_df[col].mean() / 1000
        node_ratios[node] = {
            'normal_ms': round(nv,3), 'bn_ms': round(bv,3),
            'abs_diff':  round(bv-nv,3),
            'ratio':     round(bv/nv,4) if nv>0 else 0,
            'flagged':   node in flagged_set
        }

    lat_df = df[latency_cols].copy()
    lat_df.columns = [c.replace('_latency','') for c in latency_cols]
    corr   = lat_df.corr()
    fl     = [c for c in lat_df.columns if c in flagged_set]
    unfl   = [c for c in lat_df.columns if c not in flagged_set]
    ff     = corr.loc[fl, fl].values.copy() if len(fl)>1 else np.array([[np.nan]])
    np.fill_diagonal(ff, np.nan)
    fu     = corr.loc[fl, unfl].values if (len(fl)>0 and len(unfl)>0) else np.array([[0]])

    X_lat = df[latency_cols].values
    X_res = df[cpu_cols + mem_cols + net_rx_cols + net_tx_cols].values

    feature_aucs = {}
    for name, X in [('Latency only', X_lat),
                    ('Resource only', X_res)]:
        sc = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            X, y, cv=5, scoring='roc_auc')
        feature_aucs[name] = {'mean': round(sc.mean(),4),
                              'std':  round(sc.std(),4)}

    print(f"\n[L3] Latency-only AUC: "
          f"{feature_aucs['Latency only']['mean']:.4f}  "
          f"Resource-only AUC: "
          f"{feature_aucs['Resource only']['mean']:.4f}")
    print(f"     FF corr: {np.nanmean(ff):.4f}  "
          f"FU corr: {np.nanmean(fu):.4f}")

    scaler   = StandardScaler()
    X_lat_sc = scaler.fit_transform(X_lat)
    models   = {
        'Logistic Regression': (LogisticRegression(max_iter=1000,
                                 random_state=42), X_lat_sc),
        'Random Forest':       (RandomForestClassifier(n_estimators=200,
                                 random_state=42, n_jobs=-1), X_lat),
        'Gradient Boosting':   (GradientBoostingClassifier(n_estimators=100,
                                 random_state=42), X_lat),  # fixed
    }
    model_results = {}
    for name, (model, X) in models.items():
        auc_s = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
        f1_s  = cross_val_score(model, X, y, cv=5, scoring='f1')
        model_results[name] = {'auc': round(auc_s.mean(),4),
                               'f1':  round(f1_s.mean(),4),
                               'std': round(auc_s.std(),4)}

    baseline_auc = cross_val_score(
        RandomForestClassifier(n_estimators=100,
                               random_state=42, n_jobs=-1),
        X_lat, y, cv=5, scoring='roc_auc').mean()

    ablation = {}
    for i, col in enumerate(latency_cols):
        node = col.replace('_latency','')
        sc   = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            np.delete(X_lat, i, axis=1), y, cv=3,
            scoring='roc_auc').mean()
        ablation[node] = round(sc,4)

    drops   = {n: round(baseline_auc-sc,4) for n,sc in ablation.items()}
    drops_s = sorted(drops.items(), key=lambda x: x[1], reverse=True)

    print(f"\n[L4] RF AUC: {model_results['Random Forest']['auc']:.4f}  "
          f"F1: {model_results['Random Forest']['f1']:.4f}")
    print(f"     LR AUC: "
          f"{model_results['Logistic Regression']['auc']:.4f}  "
          f"GB AUC: {model_results['Gradient Boosting']['auc']:.4f}")
    print(f"     Top ablation: {drops_s[:3]}")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Analysis: {FILE_NAME}\n'
                 f'{WORKFLOW} | {BN_TYPE} | {LOAD_RPS} RPS | Run {RUN_NUM}',
                 fontsize=12, fontweight='bold')

    ax = axes[0,0]
    ax.bar(['Normal','Bottleneck'], [n_normal, n_bottleneck],
           color=['#2196F3','#F44336'], edgecolor='white', width=0.5)
    ax.set_title('Class balance', fontweight='bold')
    ax.set_ylim(0, max(n_normal, n_bottleneck)*1.25)
    for i, val in enumerate([n_normal, n_bottleneck]):
        ax.text(i, val + max(n_normal,n_bottleneck)*0.02,
                f'{val:,}\n({val/n_total*100:.1f}%)',
                ha='center', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[0,1]
    ax.bar(range(30), [1]*30,
           color=['#F44336' if str(n) in flagged_set
                  else '#E0E0E0' for n in range(30)],
           edgecolor='white', width=0.8)
    ax.set_xticks(range(30))
    ax.set_xticklabels([str(n) for n in range(30)], fontsize=6, rotation=90)
    ax.set_yticks([])
    ax.set_title(f'Flagged nodes ({len(flagged_set)}/30)', fontweight='bold')
    ax.spines[['top','right','left']].set_visible(False)

    ax = axes[0,2]
    nodes_s  = sorted(node_ratios.keys(), key=int)
    ratios_v = [node_ratios[n]['ratio'] for n in nodes_s]
    ax.bar(range(30), ratios_v,
           color=['#F44336' if node_ratios[n]['flagged']
                  else '#90CAF9' for n in nodes_s],
           edgecolor='white', width=0.7)
    ax.axhline(1.0, color='gray', linestyle='--', lw=1, alpha=0.7)
    ax.set_xticks(range(30))
    ax.set_xticklabels(nodes_s, fontsize=6, rotation=90)
    ax.set_title('Per-node latency ratio', fontweight='bold')
    ax.set_ylabel('BN / Normal')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,0]
    rv = list(res_ratios.values())
    ax.barh(list(res_ratios.keys()), rv,
            color=['#EF5350' if v>1.1 else '#FFA726'
                   if v>1.03 else '#66BB6A' for v in rv],
            edgecolor='white')
    ax.axvline(1.0, color='gray', linestyle='--', lw=1)
    ax.set_title('Resource ratios', fontweight='bold')
    for i, v in enumerate(rv):
        ax.text(v+0.001, i, f'{v:.4f}x', va='center', fontsize=9)
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,1]
    fa_vals = [feature_aucs[k]['mean'] for k in feature_aucs]
    ax.bar(range(2), fa_vals, color=['#4CAF50','#FF9800'],
           edgecolor='white', width=0.4)
    ax.set_xticks(range(2))
    ax.set_xticklabels(['Latency\nonly','Resource\nonly'], fontsize=9)
    ax.set_ylim(0.5, 1.05)
    ax.set_title('AUC by feature set (RF)', fontweight='bold')
    for i, v in enumerate(fa_vals):
        ax.text(i, v+0.008, f'{v:.4f}', ha='center',
                fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,2]
    ax.barh(range(8), [v for _,v in drops_s[:8]][::-1],
            color=['#D32F2F' if v>0.008 else '#FF9800'
                   if v>0.004 else '#FFC107'
                   for _,v in drops_s[:8]][::-1],
            edgecolor='white')
    ax.set_yticks(range(8))
    ax.set_yticklabels([f'node {n}' for n,_ in drops_s[:8]][::-1], fontsize=8)
    ax.set_title(f'Node ablation (base={baseline_auc:.4f})', fontweight='bold')
    ax.set_xlabel('AUC drop')
    ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    chart_path = os.path.join(CHARTS_DIR, f'{FILE_ID}_analysis.png')
    plt.savefig(chart_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Chart saved: {chart_path}")

    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4), 'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
        'never_flagged': sorted(never_set, key=int),
        'n_flagged': len(flagged_set),
        'always_same_subgraph': bool(multi.nunique()==1),
        'simultaneous_rpcs': int(multi.mode()[0]),
        'entry_latency': {
            'normal_p50_ms': round(n_lat.median(),2),
            'bn_p50_ms':     round(b_lat.median(),2),
            'ratio_p50':     round(b_lat.median()/n_lat.median(),4),
            'ratio_mean':    round(b_lat.mean()/n_lat.mean(),4),
        },
        'resource_ratios': {k: round(v,4) for k,v in res_ratios.items()},
        'feature_aucs':   feature_aucs,
        'model_results':  model_results,
        'baseline_auc':   round(baseline_auc,4),
        'ablation_top8':  {n: v for n,v in drops_s[:8]},
        'node_ratios':    node_ratios,
        'ff_corr': round(float(np.nanmean(ff)),4),
        'fu_corr': round(float(np.nanmean(fu)),4),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Results saved: {results_path}")
    os.remove(local_csv)
    print(f"Local file deleted.")
    print(f"\n{'='*50}")
    print(f"✓ {FILE_ID} COMPLETE")
    print(f"{'='*50}")

## Batch 3a Results: CPU bottleneck — 800 RPS — aug30 (runs 0–9)
*Files: cpu_aug30_25min_800_[0-9]_graph_1.csv | 9 valid files | ~403,000 traces*
*Note: run 9 had only 257 traces (100% BN) — skipped modeling, flagged nodes recorded only*

### The 9-node pattern is perfectly stable across all aug30 runs

Every valid run flags exactly the same 9 nodes:
**{4, 5, 13, 14, 20, 21, 26, 27, 28}**
Zero variation across 9 runs. The pattern is as rigid as the 15-node
pattern was at 200/400 RPS.

### Quantitative summary — aug30 800 RPS

| Metric | Range across runs | Interpretation |
|--------|-------------------|----------------|
| BN rate | 29.1% – 47.1% | More variable than 200 RPS |
| Entry p50 ratio | 1.17x – 1.22x | Very weak surface signal |
| RF AUC | 0.815 – 0.847 | Significantly lower than 200/400 RPS |
| LR AUC | 0.723 – 0.766 | Also lower — boundary harder |
| Resource CPU | 1.09x – 1.66x | High but unreliable |

### Node 4 becomes the dominant node at aug30 800 RPS
At 200/400 RPS node 14 led every ablation.
At aug30 800 RPS **node 4 leads every run** (drop 0.024–0.038).
Node 8 takes second place in every run — despite not being in the
9-node flagged set. This is a key signal: node 8 carries indirect
information about the constraint even when unlabeled.

### Why is detection harder at aug30 800 RPS?
The entry point ratio is only 1.17–1.22x — barely distinguishable
from noise. The bottleneck is confined to 9 deep storage/timeline
nodes, and their latency increase is small in absolute terms. The
classifier must work much harder to find the pattern, hence lower AUC.

### Theory of Constraints — aug30 800 RPS
The 9-node core {4,5,13,14,20,21,26,27,28} represents the irreducible
constraint under CPU stress at 800 RPS. These are all storage-layer and
timeline-writing services: post-storage, user-timeline, write-home-timeline,
social-graph. When CPU is stressed at high load, only the slowest
(most resource-intensive) services remain visibly affected — the others
recover or normalize relative to the elevated baseline.

## Batch 3b Results: CPU bottleneck — 800 RPS — July files
*Files: cpu_july28, cpu_july29 (3 variants), cpu_july30 | 5 files | ~268,000 traces*

### MAJOR DISCOVERY: A completely new subgraph pattern — only 4 nodes

All five July files flag exactly the same 4 nodes:
**{0, 1, 2, 22}**

This is entirely different from anything seen before:
- Not the 15-node pattern (200/400 RPS and aug18)
- Not the 9-node pattern (aug30)
- A new 4-node pattern concentrated at the **entry path**

| Previous patterns | July pattern |
|------------------|--------------|
| 15 nodes: deep + entry path | 4 nodes: **entry path only** |
| 9 nodes: deep storage/timeline | 4 nodes: **nginx + entry chain** |

### The 4-node pattern has a dramatically stronger latency signal

| File | Entry p50 ratio | RF AUC |
|------|-----------------|--------|
| cpu_july28_800_0 | **9.35x** | 0.958 |
| cpu_july29_25min_800_0 | **6.28x** | 0.934 |
| cpu_july29_30min_800_0 | 2.20x | 0.838 |
| cpu_july29_800_0 | **6.08x** | 0.931 |
| cpu_july30_25min_800_0 | **6.86x** | 0.943 |

Entry ratios of 6–9x are the **strongest we have seen in the entire
dataset** — far stronger than the 1.3–1.7x seen in all other patterns.
The bottleneck is hitting the entry layer directly and catastrophically.

### What are nodes 0, 1, 2, and 22?
These are the **entry-path accumulation nodes** — they carry the
accumulated latency of everything downstream. Node 22 appears for
the first time as a flagged node. This suggests the CPU stress in
the July experiments was injected closer to the nginx/front-end
layer rather than in the deep storage services.

### Resource signals are strong here — but still not reliable
CPU ratio reaches 2.57x (july28) and 2.65x (july30). These are the
highest resource signals in the dataset. Yet the latency signal is
even stronger (9.35x). Interestingly, july29_800_0 shows CPU=1.02x
while the latency ratio is still 6.08x — proving again that resource
signals are experiment-dependent and cannot be relied upon alone.

### Flagged↔Unflagged correlation drops to 0.07–0.08
The 4 flagged nodes are almost completely decorrelated from the
26 unflagged nodes (FU corr = 0.07–0.08), compared to 0.20–0.38
in all previous patterns. This means the 4-node bottleneck is highly
localized — it does not propagate visibly to the rest of the system.

### Three distinct CPU bottleneck patterns confirmed

| Pattern | Flagged nodes | Files | Signal strength | AUC range |
|---------|--------------|-------|-----------------|-----------|
| Wide (15-node) | 4,5,7,8,11,12,13,14,18,19,20,21,26,27,28 | aug12, aug18 | 1.3–1.75x | 0.955–0.981 |
| Core (9-node) | 4,5,13,14,20,21,26,27,28 | aug30 | 1.17–1.22x | 0.815–0.847 |
| Entry (4-node) | 0,1,2,22 | july28/29/30 | 2.2–9.35x | 0.838–0.958 |

The 9-node core is a **subset of the 15-node wide pattern**.
The 4-node entry pattern is **completely disjoint** from both.
This tells us the bottleneck was injected on a different service
in July — targeting the entry/proxy layer rather than storage.

### Theory of Constraints — three constraint types
Under CPU stress, the system can have three different constraint locations:
1. **Entry constraint** (July): nginx and front-end chain — highest impact,
   most visible, but localized to 4 nodes
2. **Wide storage constraint** (aug18): full compose chain affected —
   15 services all show elevated latency together
3. **Core storage constraint** (aug30): only the 9 slowest storage
   services remain visibly constrained at high load

The irreducible core {4,5,13,14,20,21,26,27,28} appears in pattern 2
but not pattern 3's behavior — and the entry nodes {0,1,2} appear in
pattern 1 as unflagged carriers of accumulated delay in patterns 2&3.

### Next files to analyze
Remaining 800 RPS CPU files:
- cpu_july30_25min_repeat_800 (runs 0–2)
- cpu_july31_25min_repeat_800 (runs 0–9)
- cpu_july31_25min_rerun_800 (runs 1–7)
- cpu_sept4, cpu_sept9, cpu_sept20, cpu_sept_test files

**Key question:** Do the September files show a 4th pattern,
or do they fall into one of the three established patterns?

## Batch 3c Results: CPU bottleneck — 800 RPS — july30 repeat (runs 0–2)
*Files: cpu_july30_25min_repeat_800_[0-2]_graph_1.csv | 3 files | ~130,000 traces*

### The 4-node entry pattern holds perfectly
All three runs flag exactly the same 4 nodes: **{0, 1, 2, 22}**
This is now confirmed across 8 consecutive July files (july28, july29 ×3,
july30, july30_repeat ×3) — the entry-layer bottleneck is the defining
signature of all July experiments.

### Notable: BN rate exceeds 50% for the first time
These repeat runs show 53.8%, 57.8%, and 56.4% bottleneck rates —
the highest in the entire dataset so far. The researchers ran more
bottleneck traces than normal traces in these sessions.

### Latency signal remains strong and consistent
| Run | Entry p50 ratio | RF AUC |
|-----|-----------------|--------|
| repeat_0 | 6.81x | 0.946 |
| repeat_1 | 6.92x | 0.936 |
| repeat_2 | 7.98x | 0.937 |

The 6–8x ratio range is consistent with the other July files,
confirming the same injection configuration.

### FU correlation drops further — now 0.05–0.06
The flagged-to-unflagged correlation is now nearly zero (0.05),
the lowest in the entire dataset. The 4-node bottleneck is
completely isolated — it does not propagate to any of the 26
other nodes. This is the most localized constraint pattern we
have found.

### Node 0 dominates ablation in all three runs
Node 0 (nginx-thrift, the system entry point) takes first place
in runs 0 and 2. This confirms the bottleneck originates at or
very near the entry point of the system.

---

## Cross-batch Summary: All CPU 800 RPS files analyzed so far

### Complete pattern map

| Date group | Files | Flagged nodes | Count | Signal | AUC range |
|------------|-------|--------------|-------|--------|-----------|
| aug18 | 2 | {4,5,7,8,11,12,13,14,18,19,20,21,26,27,28} | 15 | 1.72–1.75x | 0.962–0.968 |
| aug30 | 9 | {4,5,13,14,20,21,26,27,28} | 9 | 1.17–1.22x | 0.815–0.847 |
| july28/29/30 | 8 | {0,1,2,22} | 4 | 2.2–9.35x | 0.838–0.958 |

### Three injection locations confirmed
The data is now telling us that **where** the CPU stress is injected
determines which subgraph appears:

- **Entry injection (July):** Nodes 0,1,2,22 — nginx and front-end chain
  → catastrophic latency spike (up to 9.35x), highly localized
- **Wide storage injection (aug18):** All 15 nodes — full compose chain
  → moderate spread (1.72x), high detectability
- **Core storage injection (aug30):** 9 deep nodes only
  → weak signal (1.17x), hardest to detect

### The irreducible storage core {4,5,13,14,20,21,26,27,28}
This 9-node set appears in both aug18 (as part of the 15) and aug30
(as the complete set). It never appears in July files at all.
Conclusion: these 9 nodes are the structural storage constraint —
they activate whenever storage-layer CPU stress is present.

### Still to analyze (800 RPS CPU)
- cpu_july31_repeat (runs 0–9) — likely 4-node entry pattern
- cpu_july31_rerun (runs 1–7) — likely 4-node entry pattern
- cpu_sept4, cpu_sept9 — unknown, may reveal a 4th pattern
- cpu_sept20 — unknown
- cpu_test_july24 — unknown

---
## Batch 3d: CPU bottleneck — 800 RPS — july31 files
*Files: cpu_july31_25min_repeat_800_[0-9] then cpu_july31_25min_rerun_800_[1-7]*
*Question: Does the 4-node pattern hold through all of July?*

## Batch 3d Results: CPU bottleneck — 800 RPS — july31 repeat (runs 0–9)
*Files: cpu_july31_25min_repeat_800_[0-9]_graph_1.csv | 10 files*

### Only run 0 had bottleneck traces — runs 1–9 are all baseline-only

| Run | Status | Traces | BN rate |
|-----|--------|--------|---------|
| 0 | ✓ Valid | 36,088 | 54.4% |
| 1 | ⚠ Baseline only | 56,365 | 0.0% |
| 2 | ⚠ Baseline only | 31,029 | 0.0% |
| 3 | ⚠ Baseline only | 41,855 | 0.0% |
| 4 | ⚠ Baseline only | 48,850 | 0.0% |
| 5 | ⚠ Baseline only | 38,771 | 0.0% |
| 6 | ⚠ Baseline only | 3,303 | 0.0% |
| 7 | ⚠ Baseline only | 28,615 | 0.0% |
| 8 | ⚠ Baseline only | 44,443 | 0.0% |
| 9 | ⚠ Baseline only | 10,505 | 0.0% |

Runs 1–9 represent a long baseline recording session — the researchers
were measuring the system without any stress injection. This is valuable
as a clean 800 RPS baseline reference: p50 latency ranges from
19.6ms to 22.4ms across runs, very close to the 200/400 RPS baselines.
The system baseline is remarkably stable across load levels.

### Run 0: 4-node entry pattern confirmed again
Flagged nodes: **{0, 1, 2, 22}** — identical to all other July files.
Entry p50 ratio = 5.10x. RF AUC = 0.899.
Node 0 leads ablation again, confirming it as the most informative
single node for this entry-layer constraint type.

### Baseline latency summary across all baseline-only files so far
| Load | Files | p50 range |
|------|-------|-----------|
| 400 RPS | aug9 runs 0-2 | 17.3–17.5ms |
| 800 RPS | july31 runs 1-9 | 19.6–22.4ms |

The 800 RPS baseline is only slightly higher than 400 RPS (2–5ms more).
This confirms the system is not saturated even at 800 RPS under
normal conditions — the bottlenecks we detect are injection-driven,
not load-driven saturation.

---
## Batch 3e: CPU bottleneck — 800 RPS — july31 rerun files
*Files: cpu_july31_25min_rerun_800_[1-7]_graph_1.csv*
*Expected: 4-node entry pattern based on all July evidence*

## Batch 3e Results: CPU bottleneck — 800 RPS — july31 rerun (runs 1–7)
*Files: cpu_july31_25min_rerun_800_[1-7]_graph_1.csv | 6 valid + 1 baseline*

### The 4-node entry pattern holds without exception
All 6 valid runs flag exactly **{0, 1, 2, 22}** — the same 4 nodes
across every single July file analyzed. Run 7 is baseline-only.
This pattern has now appeared in **16 consecutive July files** with
zero deviation in the flagged node set.

### AUC varies significantly within the 4-node pattern
| Run | Entry p50 ratio | RF AUC | Note |
|-----|-----------------|--------|------|
| rerun_1 | 6.23x | 0.942 | Strong signal |
| rerun_2 | 6.68x | 0.949 | Strong signal |
| rerun_3 | 4.99x | **0.790** | Weakest in July group |
| rerun_4 | 6.13x | 0.927 | Strong signal |
| rerun_5 | 3.89x | 0.891 | Moderate signal |
| rerun_6 | 2.33x | 0.876 | Weakest signal |

**Run 3 is an anomaly** — RF AUC drops to 0.790, the lowest in all
July files. Yet the flagged set is identical. Looking at resource
signals: CPU=0.80x — resources are actually *lower* during bottleneck
traces than normal. This is a paradox similar to node 26 seen earlier.
The injection may have been lighter intensity in this run, producing
a weaker latency separation.

**Run 6** shows BN rate of 60.5% — the highest in the dataset so far.
More than 3 in every 5 requests were bottlenecked.

### Node 0 leads ablation in 4 of 6 runs
Node 23 takes first place in runs 4 and 6. Node 23 is an unflagged
node that consistently appears in ablation top-3 across all July files.
It carries indirect signal about the entry-layer constraint even though
it is never directly labeled as bottlenecked. This suggests node 23
is structurally adjacent to the constraint path.

### FU correlation remains near zero: 0.05–0.09
Across all 16 July files the flagged-to-unflagged correlation stays
between 0.05 and 0.09. The entry bottleneck is completely isolated
from the rest of the system — a localized constraint that does not
propagate downstream. This is the defining signature of the
entry-layer injection pattern.

---
## Complete July group summary

| Metric | Range across all 16 valid July files |
|--------|--------------------------------------|
| Flagged nodes | Always {0, 1, 2, 22} — 100% consistent |
| Entry p50 ratio | 2.20x – 9.35x |
| RF AUC | 0.790 – 0.958 |
| FU correlation | 0.045 – 0.091 |
| Resource CPU | 0.80x – 2.65x (highly variable) |

The 4-node entry pattern is the most stable subgraph in the entire
dataset — 16 files, one date group, zero variation in the constraint
location. The only variation is in signal intensity (ratio and AUC),
not in which nodes are affected.

---
## Batch 3f: Remaining CPU 800 RPS files — September group
*Files: cpu_sept4, cpu_sept9, cpu_sept20, cpu_test_july24*
*Question: Do September files show a new pattern, or match one of the three established patterns?*
*Prediction: likely to match aug30 (9-node) or aug18 (15-node) based on dates*

## Batch 3f Results: CPU bottleneck — 800 RPS — sept4 (runs 0–9)
*Files: cpu_sept4_25min_800_[0-9]_graph_1.csv | 9 valid + 1 too small*

### September 4 returns to the 15-node wide pattern
All 9 valid runs flag exactly **{4,5,7,8,11,12,13,14,18,19,20,21,26,27,28}**
— the same 15 nodes as aug12 (200 RPS), aug9 (400 RPS), and aug18 (800 RPS).

This confirms that the 15-node pattern is not load-dependent —
it reappears at 800 RPS when the injection targets the storage/compose chain.

### Pattern map now complete for CPU 800 RPS

| Date | Pattern | Nodes | Signal |
|------|---------|-------|--------|
| aug18 | Wide storage | 15 | 1.51–1.75x |
| aug30 | Core storage | 9 | 1.17–1.22x |
| july28–31 | Entry layer | 4 | 2.2–9.35x |
| **sept4** | **Wide storage** | **15** | **1.49–1.76x** |

Sept4 matches aug18 almost exactly — same 15 nodes, same AUC range
(0.864–0.959), same signal strength (1.49–1.76x).

### Quantitative summary — sept4 800 RPS
| Metric | Range | Notes |
|--------|-------|-------|
| BN rate | 19.5% – 84.3% | Most variable batch yet |
| Entry p50 ratio | 1.49x – 1.76x | Consistent with aug18 |
| RF AUC | 0.864 – 0.959 | Run 5 lowest (small sample) |
| Resource CPU | 0.82x – 1.25x | Unreliable as always |

### Run 5 and 6 are anomalous — very small samples
Run 5: 747 traces (84.3% BN) — extremely unbalanced, AUC may be unreliable.
Run 6: 261 traces — too small for modeling, flagged nodes still recorded.
These short sessions suggest the recording was cut off or restarted.
Despite the small sample, both still show exactly 15 flagged nodes.

### Node 14 returns to top ablation rank
After node 4 dominated in aug30 and node 0 dominated in July files,
**node 14 leads ablation again** in 6 of 9 valid sept4 runs.
Node 4 takes second place consistently.
This matches the 200/400 RPS pattern — confirming that the 15-node
wide-storage injection has node 14 as its most informative signal.

### Theory of Constraints — sept4
The CPU stress in sept4 targets the same location as aug12/aug18:
the full storage/compose subgraph. The constraint is the 15-node
wide storage path. Node 14 (post-storage-mongodb) is the single
node whose removal most degrades detection — suggesting it sits
at the deepest, most critical point of the constraint chain.

---
## CPU Bottleneck — All 800 RPS files: Final pattern summary

Three injection configurations confirmed across all dates:

**Pattern A — Wide storage (15 nodes): {4,5,7,8,11,12,13,14,18,19,20,21,26,27,28}**
- Files: aug12 (200/400 RPS), aug18 (800 RPS), sept4 (800 RPS)
- Signal: 1.31–1.76x | AUC: 0.864–0.981
- Most important node: 14 (post-storage-mongodb)

**Pattern B — Core storage (9 nodes): {4,5,13,14,20,21,26,27,28}**
- Files: aug30 (800 RPS)
- Signal: 1.17–1.22x | AUC: 0.815–0.847
- Most important node: 4 (post-storage-service)
- Hardest to detect — weakest signal in dataset

**Pattern C — Entry layer (4 nodes): {0,1,2,22}**
- Files: july28, july29, july30, july31 (800 RPS)
- Signal: 2.2–9.35x | AUC: 0.790–0.958
- Most important node: 0 (nginx-thrift)
- Most localized — FU correlation near zero

Pattern B is a subset of Pattern A — the 9 core nodes appear in both.
Pattern C is completely disjoint from both A and B.

---
## Next: Remaining CPU 800 RPS files
*cpu_sept9, cpu_sept20, cpu_test_july24*
*Then: Memory bottleneck group (mem_sep22, mem_sep25)*

## Batch 3g Results: CPU bottleneck — 800 RPS — sept9, sept20, test_july24
*Files: cpu_sept9 (×2), cpu_sept20 (×9), cpu_test_july24 (×1) | 12 files*

### A FOURTH pattern discovered: 11-node hybrid (sept9 + sept20)

Both sept9 runs and all 9 sept20 runs flag exactly:
**{0, 1, 2, 13, 14, 20, 21, 22, 26, 27, 28}** — 11 nodes, rigid 100%.

This is a hybrid of Pattern A (entry nodes 0,1,2,22) and Pattern B
(core storage nodes 13,14,20,21,26,27,28). It combines the entry layer
with the storage core — but without the 6 middle nodes
{4,5,7,8,11,12,18,19} that appear in Pattern A.

### Complete CPU bottleneck pattern taxonomy — now 4 patterns

| Pattern | Name | Nodes | Count | Files | Signal |
|---------|------|-------|-------|-------|--------|
| A | Wide storage | {4,5,7,8,11,12,13,14,18,19,20,21,26,27,28} | 15 | aug12/18, sept4 | 1.3–1.8x |
| B | Core storage | {4,5,13,14,20,21,26,27,28} | 9 | aug30 | 1.2x |
| C | Entry layer | {0,1,2,22} | 4 | all July | 2–9x |
| **D** | **Entry+Core hybrid** | **{0,1,2,13,14,20,21,22,26,27,28}** | **11** | **sept9, sept20** | **5–7x** |

Pattern D = Pattern C entry nodes {0,1,2,22} + Pattern B storage core
{13,14,20,21,26,27,28}. The middle chain {4,5,7,8,11,12,18,19} does
not activate — only the two extremes of the call chain are affected.

### Pattern D signal strength: strong at both ends
| Metric | Range (sept9 + sept20) |
|--------|----------------------|
| Entry p50 ratio | 4.67x – 6.72x |
| RF AUC | 0.906 – 0.959 |
| LR AUC | 0.834 – 0.946 |

Signal is strong — closer to Pattern C (July) than Pattern A (aug18/sept4).
The bottleneck hits both the entry layer and the storage core simultaneously,
bypassing the middle processing chain entirely.

### Node 0 and node 22 dominate ablation — every single run
In all 11 sept9/sept20 runs: node 0 takes first or second place,
node 22 takes first or second place. Node 14 appears in third place
for sept20 files. This is consistent with Pattern D being a combination
of entry (nodes 0,22) and storage (node 14).

### FU correlation is unique for Pattern D: 0.14–0.29
Pattern C (July): FU corr = 0.05–0.09 (almost isolated)
Pattern D (sept9/20): FU corr = 0.14–0.29 (moderate coupling)
Pattern A (aug12/18): FU corr = 0.20–0.33 (moderate coupling)

Pattern D is more coupled to the rest of the system than Pattern C
but less than Pattern A — consistent with it spanning two disjoint
regions of the call chain.

### cpu_test_july24: Pattern C confirmed, but very weak
File: cpu_test_july24_800_0 — only 2.1% BN rate (786 out of 37,381).
Flagged nodes: {0,1,2,22} — Pattern C entry layer.
But signal is very weak: entry ratio only 1.55x, AUC=0.806, F1=0.389.
Resource signals: CPU=1.000x, Memory=1.000x, NetRX=1.000x — exactly
1.000 meaning the bottleneck was so rare and brief it left no resource
footprint. This appears to be an early test run with minimal injection.
Despite the near-zero signal, the correct 4-node subgraph is still
detected, showing the model's robustness even at 2% bottleneck rate.

---
## Final CPU Bottleneck Pattern Map — All files complete

**Four injection configurations across 79 analyzed CPU files:**

| Pattern | Trigger condition | Constraint location |
|---------|------------------|---------------------|
| A (15 nodes) | Storage-layer injection, moderate load | Full compose→storage chain |
| B (9 nodes) | Storage-layer injection, saturated system | Deep storage only |
| C (4 nodes) | Entry-layer injection | nginx→front-end chain |
| D (11 nodes) | Dual injection: entry + storage | Both ends, middle bypassed |

**The irreducible storage core {13,14,20,21,26,27,28} appears in
patterns A, B, and D** — confirming it as the structural constraint
of the storage layer under any CPU stress configuration.

**Node 0 (nginx-thrift) is the most important node in patterns C and D.**
**Node 14 (post-storage-mongodb) is the most important node in patterns A and B.**
These two nodes are the primary constraint indicators for their
respective injection locations.

---
## Next batch: Memory bottleneck group
*Files: mem_sep22_10min_800_[0-9] + mem_sep25_10min_800_[0-29]*
*Key question: Does memory stress activate the same subgraphs as CPU stress,
or does it create entirely different constraint patterns?*
*This is the first cross-bottleneck-type comparison in the analysis.*

In [ ]:
# ============================================================
# CELL 5 — Corrected (re-run for batch 4)
# ============================================================

FILE_NAME   = 'cpu_july28_800_0_graph_1.csv'
BN_TYPE     = 'CPU stress'
WORKFLOW    = 'Compose'
LOAD_RPS    = 800
RUN_NUM     = 0

KAGGLE_PATH = (
    'processed_dataset/compose/multi-modal-data-separate/'
    + FILE_NAME
)
FILE_ID = (FILE_NAME
           .replace('_graph_1.csv','')
           .replace('_25min_repeat','')
           .replace('_25min_rerun','')
           .replace('_25min','')
           .replace('_30min',''))

LOCAL_DIR = '/content/ms_data'
local_csv = os.path.join(LOCAL_DIR, FILE_NAME)
os.makedirs(LOCAL_DIR, exist_ok=True)

if not os.path.exists(local_csv):
    print(f"Downloading {FILE_NAME}...")
    result = subprocess.run([
        'kaggle', 'datasets', 'download',
        'gagansomashekar/microservices-bottleneck-detection-dataset',
        '--path', LOCAL_DIR, '--file', KAGGLE_PATH
    ], capture_output=True, text=True)
    for f in os.listdir(LOCAL_DIR):
        if f.endswith('.zip'):
            with zipfile.ZipFile(os.path.join(LOCAL_DIR, f)) as z:
                z.extractall(LOCAL_DIR)
            os.remove(os.path.join(LOCAL_DIR, f))
    print("Downloaded and unzipped.")
else:
    print(f"Already exists: {FILE_NAME}")

df = pd.read_csv(local_csv)

latency_cols = [c for c in df.columns if c.endswith('_latency')]
rpc_cols     = [c for c in df.columns if c.endswith('_label_RPC')]
cpu_cols     = [c for c in df.columns if 'cpu' in c and 'label' not in c]
mem_cols     = [c for c in df.columns if 'memory' in c]
net_rx_cols  = [c for c in df.columns if 'receive' in c]
net_tx_cols  = [c for c in df.columns if 'transmit' in c]

n_total      = len(df)
n_normal     = int((df['label_trace']==0).sum())
n_bottleneck = int((df['label_trace']==1).sum())
bn_rate      = n_bottleneck / n_total
normal_df    = df[df['label_trace']==0]
bottleneck_df= df[df['label_trace']==1]
y            = df['label_trace'].values

print(f"\nLoaded: {FILE_NAME}")
print(f"  {n_total:,} traces | Normal: {n_normal:,} | "
      f"BN: {n_bottleneck:,} | Rate: {bn_rate*100:.1f}%")

MIN_TRACES   = 500
BOTH_CLASSES = (n_normal > 0 and n_bottleneck > 0)
ENOUGH_DATA  = (n_total >= MIN_TRACES)

if not BOTH_CLASSES:
    status = 'BASELINE_ONLY' if n_bottleneck == 0 else 'BOTTLENECK_ONLY'
elif not ENOUGH_DATA:
    status = 'TOO_SMALL'
else:
    status = 'OK'

if status != 'OK':
    print(f"\n⚠ Skipping modeling — status: {status}")
    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    n_lat = normal_df['0_latency']/1000 if n_normal > 0 else None
    b_lat = bottleneck_df['0_latency']/1000 if n_bottleneck > 0 else None
    print(f"  Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    if n_lat is not None:
        print(f"  Normal p50: {n_lat.median():.1f}ms")
    if b_lat is not None:
        print(f"  BN p50: {b_lat.median():.1f}ms")
    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4),
        'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  Saved: {results_path}")
    os.remove(local_csv)
    print(f"  Local file deleted.")
    print(f"\n{'='*50}")
    print(f"⚠ {FILE_ID} — {status}")
    print(f"{'='*50}")

else:
    n_lat = normal_df['0_latency'] / 1000
    b_lat = bottleneck_df['0_latency'] / 1000

    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    never_set   = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates==0].index)
    df['n_bn_rpcs'] = df[rpc_cols].sum(axis=1)
    multi = df[df['label_trace']==1]['n_bn_rpcs']

    res_ratios = {
        'cpu':    bottleneck_df[cpu_cols].mean().mean() /
                  normal_df[cpu_cols].mean().mean(),
        'memory': bottleneck_df[mem_cols].mean().mean() /
                  normal_df[mem_cols].mean().mean(),
        'net_rx': bottleneck_df[net_rx_cols].mean().mean() /
                  normal_df[net_rx_cols].mean().mean(),
        'net_tx': bottleneck_df[net_tx_cols].mean().mean() /
                  normal_df[net_tx_cols].mean().mean(),
    }

    print(f"\n[L2] Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    print(f"     Never flagged ({len(never_set)}): "
          f"{sorted(never_set, key=int)}")
    print(f"     Simultaneous RPCs: always {int(multi.mode()[0])} "
          f"| Rigid: {multi.nunique()==1}")
    print(f"     Entry p50 — Normal: {n_lat.median():.1f}ms  "
          f"BN: {b_lat.median():.1f}ms  "
          f"Ratio: {b_lat.median()/n_lat.median():.2f}x")
    print(f"     Resources: CPU={res_ratios['cpu']:.4f}x  "
          f"Mem={res_ratios['memory']:.4f}x  "
          f"NetRX={res_ratios['net_rx']:.4f}x")

    node_ratios = {}
    for col in latency_cols:
        node = col.replace('_latency','')
        nv   = normal_df[col].mean() / 1000
        bv   = bottleneck_df[col].mean() / 1000
        node_ratios[node] = {
            'normal_ms': round(nv,3), 'bn_ms': round(bv,3),
            'abs_diff':  round(bv-nv,3),
            'ratio':     round(bv/nv,4) if nv>0 else 0,
            'flagged':   node in flagged_set
        }

    lat_df = df[latency_cols].copy()
    lat_df.columns = [c.replace('_latency','') for c in latency_cols]
    corr   = lat_df.corr()
    fl     = [c for c in lat_df.columns if c in flagged_set]
    unfl   = [c for c in lat_df.columns if c not in flagged_set]
    ff     = corr.loc[fl, fl].values.copy() if len(fl)>1 else np.array([[np.nan]])
    np.fill_diagonal(ff, np.nan)
    fu     = corr.loc[fl, unfl].values if (len(fl)>0 and len(unfl)>0) else np.array([[0]])

    X_lat = df[latency_cols].values
    X_res = df[cpu_cols + mem_cols + net_rx_cols + net_tx_cols].values

    feature_aucs = {}
    for name, X in [('Latency only', X_lat),
                    ('Resource only', X_res)]:
        sc = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            X, y, cv=5, scoring='roc_auc')
        feature_aucs[name] = {'mean': round(sc.mean(),4),
                              'std':  round(sc.std(),4)}

    print(f"\n[L3] Latency-only AUC: "
          f"{feature_aucs['Latency only']['mean']:.4f}  "
          f"Resource-only AUC: "
          f"{feature_aucs['Resource only']['mean']:.4f}")
    print(f"     FF corr: {np.nanmean(ff):.4f}  "
          f"FU corr: {np.nanmean(fu):.4f}")

    scaler   = StandardScaler()
    X_lat_sc = scaler.fit_transform(X_lat)
    models   = {
        'Logistic Regression': (LogisticRegression(max_iter=1000,
                                 random_state=42), X_lat_sc),
        'Random Forest':       (RandomForestClassifier(n_estimators=200,
                                 random_state=42, n_jobs=-1), X_lat),
        'Gradient Boosting':   (GradientBoostingClassifier(n_estimators=100,
                                 random_state=42), X_lat),  # fixed
    }
    model_results = {}
    for name, (model, X) in models.items():
        auc_s = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
        f1_s  = cross_val_score(model, X, y, cv=5, scoring='f1')
        model_results[name] = {'auc': round(auc_s.mean(),4),
                               'f1':  round(f1_s.mean(),4),
                               'std': round(auc_s.std(),4)}

    baseline_auc = cross_val_score(
        RandomForestClassifier(n_estimators=100,
                               random_state=42, n_jobs=-1),
        X_lat, y, cv=5, scoring='roc_auc').mean()

    ablation = {}
    for i, col in enumerate(latency_cols):
        node = col.replace('_latency','')
        sc   = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            np.delete(X_lat, i, axis=1), y, cv=3,
            scoring='roc_auc').mean()
        ablation[node] = round(sc,4)

    drops   = {n: round(baseline_auc-sc,4) for n,sc in ablation.items()}
    drops_s = sorted(drops.items(), key=lambda x: x[1], reverse=True)

    print(f"\n[L4] RF AUC: {model_results['Random Forest']['auc']:.4f}  "
          f"F1: {model_results['Random Forest']['f1']:.4f}")
    print(f"     LR AUC: "
          f"{model_results['Logistic Regression']['auc']:.4f}  "
          f"GB AUC: {model_results['Gradient Boosting']['auc']:.4f}")
    print(f"     Top ablation: {drops_s[:3]}")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Analysis: {FILE_NAME}\n'
                 f'{WORKFLOW} | {BN_TYPE} | {LOAD_RPS} RPS | Run {RUN_NUM}',
                 fontsize=12, fontweight='bold')

    ax = axes[0,0]
    ax.bar(['Normal','Bottleneck'], [n_normal, n_bottleneck],
           color=['#2196F3','#F44336'], edgecolor='white', width=0.5)
    ax.set_title('Class balance', fontweight='bold')
    ax.set_ylim(0, max(n_normal, n_bottleneck)*1.25)
    for i, val in enumerate([n_normal, n_bottleneck]):
        ax.text(i, val + max(n_normal,n_bottleneck)*0.02,
                f'{val:,}\n({val/n_total*100:.1f}%)',
                ha='center', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[0,1]
    ax.bar(range(30), [1]*30,
           color=['#F44336' if str(n) in flagged_set
                  else '#E0E0E0' for n in range(30)],
           edgecolor='white', width=0.8)
    ax.set_xticks(range(30))
    ax.set_xticklabels([str(n) for n in range(30)], fontsize=6, rotation=90)
    ax.set_yticks([])
    ax.set_title(f'Flagged nodes ({len(flagged_set)}/30)', fontweight='bold')
    ax.spines[['top','right','left']].set_visible(False)

    ax = axes[0,2]
    nodes_s  = sorted(node_ratios.keys(), key=int)
    ratios_v = [node_ratios[n]['ratio'] for n in nodes_s]
    ax.bar(range(30), ratios_v,
           color=['#F44336' if node_ratios[n]['flagged']
                  else '#90CAF9' for n in nodes_s],
           edgecolor='white', width=0.7)
    ax.axhline(1.0, color='gray', linestyle='--', lw=1, alpha=0.7)
    ax.set_xticks(range(30))
    ax.set_xticklabels(nodes_s, fontsize=6, rotation=90)
    ax.set_title('Per-node latency ratio', fontweight='bold')
    ax.set_ylabel('BN / Normal')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,0]
    rv = list(res_ratios.values())
    ax.barh(list(res_ratios.keys()), rv,
            color=['#EF5350' if v>1.1 else '#FFA726'
                   if v>1.03 else '#66BB6A' for v in rv],
            edgecolor='white')
    ax.axvline(1.0, color='gray', linestyle='--', lw=1)
    ax.set_title('Resource ratios', fontweight='bold')
    for i, v in enumerate(rv):
        ax.text(v+0.001, i, f'{v:.4f}x', va='center', fontsize=9)
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,1]
    fa_vals = [feature_aucs[k]['mean'] for k in feature_aucs]
    ax.bar(range(2), fa_vals, color=['#4CAF50','#FF9800'],
           edgecolor='white', width=0.4)
    ax.set_xticks(range(2))
    ax.set_xticklabels(['Latency\nonly','Resource\nonly'], fontsize=9)
    ax.set_ylim(0.5, 1.05)
    ax.set_title('AUC by feature set (RF)', fontweight='bold')
    for i, v in enumerate(fa_vals):
        ax.text(i, v+0.008, f'{v:.4f}', ha='center',
                fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,2]
    ax.barh(range(8), [v for _,v in drops_s[:8]][::-1],
            color=['#D32F2F' if v>0.008 else '#FF9800'
                   if v>0.004 else '#FFC107'
                   for _,v in drops_s[:8]][::-1],
            edgecolor='white')
    ax.set_yticks(range(8))
    ax.set_yticklabels([f'node {n}' for n,_ in drops_s[:8]][::-1], fontsize=8)
    ax.set_title(f'Node ablation (base={baseline_auc:.4f})', fontweight='bold')
    ax.set_xlabel('AUC drop')
    ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    chart_path = os.path.join(CHARTS_DIR, f'{FILE_ID}_analysis.png')
    plt.savefig(chart_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Chart saved: {chart_path}")

    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4), 'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
        'never_flagged': sorted(never_set, key=int),
        'n_flagged': len(flagged_set),
        'always_same_subgraph': bool(multi.nunique()==1),
        'simultaneous_rpcs': int(multi.mode()[0]),
        'entry_latency': {
            'normal_p50_ms': round(n_lat.median(),2),
            'bn_p50_ms':     round(b_lat.median(),2),
            'ratio_p50':     round(b_lat.median()/n_lat.median(),4),
            'ratio_mean':    round(b_lat.mean()/n_lat.mean(),4),
        },
        'resource_ratios': {k: round(v,4) for k,v in res_ratios.items()},
        'feature_aucs':   feature_aucs,
        'model_results':  model_results,
        'baseline_auc':   round(baseline_auc,4),
        'ablation_top8':  {n: v for n,v in drops_s[:8]},
        'node_ratios':    node_ratios,
        'ff_corr': round(float(np.nanmean(ff)),4),
        'fu_corr': round(float(np.nanmean(fu)),4),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Results saved: {results_path}")
    os.remove(local_csv)
    print(f"Local file deleted.")
    print(f"\n{'='*50}")
    print(f"✓ {FILE_ID} COMPLETE")
    print(f"{'='*50}")

## Batch 4a Results: Memory bottleneck — 800 RPS — sep22 (runs 0–9)
*Files: mem_sep22_10min_800_[0-9]_graph_1.csv | 10 files | ~159,000 traces*
*First memory bottleneck batch — cross-type comparison with CPU*

### CRITICAL FINDING: Memory stress activates the SAME 11-node subgraph as CPU Pattern D

All 10 runs flag exactly **{0,1,2,13,14,20,21,22,26,27,28}** — identical
to the CPU sept9 and sept20 files (Pattern D: Entry+Core hybrid).

This is the first cross-bottleneck-type result in the analysis.
Memory stress does not create a unique subgraph — it activates the
same structural constraint path as CPU Pattern D.

### Memory vs CPU Pattern D — direct comparison

| Metric | CPU Pattern D (sept9/20) | Memory sep22 |
|--------|--------------------------|--------------|
| Flagged nodes | {0,1,2,13,14,20,21,22,26,27,28} | **identical** |
| Entry p50 ratio | 4.67–6.72x | 2.56–4.54x |
| RF AUC | 0.906–0.959 | **0.635–0.888** |
| FU correlation | 0.14–0.29 | 0.33–0.38 |

Memory stress produces the same subgraph but **weaker signal** and
**lower AUC** than CPU Pattern D. Detection is harder under memory
stress — the latency increase is smaller and less consistent.

### AUC is highly variable — worst batch in the dataset so far
| Run | RF AUC | Entry ratio |
|-----|--------|-------------|
| 0 | 0.783 | 3.93x |
| 1 | 0.888 | 4.09x |
| 2 | 0.801 | 3.61x |
| 3 | **0.636** | 2.56x |
| 4 | 0.831 | 3.29x |
| 5 | 0.825 | 2.83x |
| 6 | **0.678** | 2.74x |
| 7 | 0.857 | 4.54x |
| 8 | 0.821 | 3.87x |
| 9 | 0.795 | 3.01x |

Runs 3 and 6 are the weakest — AUC 0.636 and 0.678. Run 3 is
remarkable: normal baseline p50 = 42.7ms (vs 21–29ms in other runs),
suggesting the system was already under load before injection began,
making bottleneck traces harder to distinguish from normal ones.

### Memory ratio paradox: Mem ratio < 1.0 in most runs
CPU ratio: 1.07–1.36x (elevated, as expected)
**Memory ratio: 0.91–1.02x** (at or below normal in 9 of 10 runs)

Memory stress increases CPU and network utilization but the
memory consumption metric itself does not rise — or even drops.
This is counterintuitive: memory pressure causes the OS to swap
or throttle, which shows up as CPU overhead and latency, not as
raw memory consumption increase. The memory metric measures
usage, not pressure. This is an important finding for the paper.

### Node 0 leads ablation in 9 of 10 runs
Node 0 (nginx-thrift) is the single most informative node for
memory bottleneck detection, with drops of 0.002–0.092.
Run 2 shows an exceptionally large drop of 0.092 for node 0
and 0.077 for node 1 — unusually high, suggesting these two
runs had very concentrated signal at the entry nodes.

Negative ablation values appear in several runs (e.g. run 9:
node 16 drop = -0.010). A negative drop means removing that
node *improves* AUC — the node is adding noise. This is more
common in memory bottleneck files than CPU files, consistent
with the weaker, noisier signal under memory stress.

### FU correlation is higher than any CPU pattern: 0.33–0.38
Memory stress causes more cross-node coupling than CPU stress.
The 11 flagged nodes leak more signal to the 19 unflagged nodes
compared to CPU patterns. This suggests memory pressure
propagates more diffusely through the system — it does not stay
confined to the constraint subgraph as cleanly as CPU stress.

### Theory of Constraints — Memory vs CPU
Memory stress and CPU Pattern D activate the **same constraint path**
{0,1,2,13,14,20,21,22,26,27,28}. This means the system's structural
vulnerability is not resource-type-specific — the same services are
the constraint regardless of whether CPU or memory is stressed.
The constraint is architectural (call chain structure) not
resource-specific. This is a strong ToC finding.

---
## Next: Memory bottleneck — sep25 (runs 0–29)
*30 additional memory stress files — largest single batch in the dataset*
*Question: Does the 11-node pattern hold across all 30 runs?*

## Batch 4b Results: Memory bottleneck — 800 RPS — sep25 (runs 0–29)
*Files: mem_sep25_10min_800_[0-29]_graph_1.csv | 30 files | ~390,000 traces*

### Pattern D holds across all 30 runs — zero exceptions
Every file flags exactly the same 11 nodes:
**{0, 1, 2, 13, 14, 20, 21, 22, 26, 27, 28}**

Pattern D is now confirmed across:
- 11 CPU files (sept9 + sept20)
- 40 Memory files (sep22 + sep25)
- **Total: 51 independent experiments — same pattern every time**

### Quantitative summary — sep25 full batch

| Metric | Range | Note |
|--------|-------|------|
| BN rate | 48.3% – 68.3% | Relatively stable |
| Entry p50 ratio | 1.55× – 4.48× | Wide variation |
| RF AUC | **0.673 – 0.901** | Widest AUC range in the entire dataset |
| LR AUC | 0.699 – 0.876 | LR performance close to RF here |
| Memory ratio | 0.896× – 1.061× | Below 1.0 in 20 of 30 files |

### Memory bottleneck is the hardest to detect in the entire dataset
Average RF AUC comparison across all patterns:

| Pattern | Bottleneck type | Mean RF AUC |
|---------|----------------|-------------|
| A — 15 nodes | CPU | ~0.970 |
| C — 4 nodes | CPU | ~0.920 |
| D — 11 nodes | CPU | ~0.935 |
| B — 9 nodes | CPU | ~0.830 |
| D — 11 nodes | **Memory sep22** | ~0.810 |
| D — 11 nodes | **Memory sep25** | **~0.800** |

Memory stress produces the weakest signal — even harder to detect
than Pattern B (CPU), which was previously considered the hardest.

### Negative ablation values are common — 15 of 30 files
Removing a node *improves* AUC in half the files. In CPU files this
was rare. This means memory bottleneck signal is diffuse and noisy —
individual nodes add noise rather than signal to the classifier.

### Two exceptional outliers

**Run 27:** Largest ablation values in the entire dataset:
node 15 drop = **0.186**, node 4 drop = 0.186, node 22 drop = 0.180.
Signal is abnormally concentrated in 3 nodes rather than distributed.

**Run 28:** Weakest performance in the batch — AUC 0.673.
Normal p50 = 34.9ms (already elevated before injection),
making bottleneck traces harder to distinguish from normal ones.

### Complete Memory group summary (sep22 + sep25 = 40 files)

| Metric | Full range |
|--------|-----------|
| Flagged nodes | Always {0,1,2,13,14,20,21,22,26,27,28} — 100% consistent |
| RF AUC | 0.635 – 0.901 |
| Entry p50 ratio | 1.55× – 4.54× |
| Memory ratio | 0.896× – 1.061× — **never rises reliably** |
| FU correlation | 0.32 – 0.41 — highest of all CPU patterns |

### Theory of Constraints implication
The deepest finding from 40 memory experiments: **the constraint is
structural, not resource-specific**. Memory stress activates the same
11-node subgraph as CPU Pattern D. The bottleneck location is determined
by the service call graph architecture, not by which resource is stressed.

---
## Batch 5a: CPU+Memory combined — sep29 (runs 0–8)
*Files: cpu_mem_sep29_10min_800_[0-8]_graph_1.csv*
*BN_TYPE = 'CPU+Memory stress' | LOAD_RPS = 800*
*Question: Does combined stress produce a new pattern, or match Pattern D?*

In [ ]:
# ============================================================
# CELL 5 — Corrected (re-run for batch 5)
# ============================================================

FILE_NAME   = 'cpu_july28_800_0_graph_1.csv'
BN_TYPE     = 'CPU stress'
WORKFLOW    = 'Compose'
LOAD_RPS    = 800
RUN_NUM     = 0

KAGGLE_PATH = (
    'processed_dataset/compose/multi-modal-data-separate/'
    + FILE_NAME
)
FILE_ID = (FILE_NAME
           .replace('_graph_1.csv','')
           .replace('_25min_repeat','')
           .replace('_25min_rerun','')
           .replace('_25min','')
           .replace('_30min',''))

LOCAL_DIR = '/content/ms_data'
local_csv = os.path.join(LOCAL_DIR, FILE_NAME)
os.makedirs(LOCAL_DIR, exist_ok=True)

if not os.path.exists(local_csv):
    print(f"Downloading {FILE_NAME}...")
    result = subprocess.run([
        'kaggle', 'datasets', 'download',
        'gagansomashekar/microservices-bottleneck-detection-dataset',
        '--path', LOCAL_DIR, '--file', KAGGLE_PATH
    ], capture_output=True, text=True)
    for f in os.listdir(LOCAL_DIR):
        if f.endswith('.zip'):
            with zipfile.ZipFile(os.path.join(LOCAL_DIR, f)) as z:
                z.extractall(LOCAL_DIR)
            os.remove(os.path.join(LOCAL_DIR, f))
    print("Downloaded and unzipped.")
else:
    print(f"Already exists: {FILE_NAME}")

df = pd.read_csv(local_csv)

latency_cols = [c for c in df.columns if c.endswith('_latency')]
rpc_cols     = [c for c in df.columns if c.endswith('_label_RPC')]
cpu_cols     = [c for c in df.columns if 'cpu' in c and 'label' not in c]
mem_cols     = [c for c in df.columns if 'memory' in c]
net_rx_cols  = [c for c in df.columns if 'receive' in c]
net_tx_cols  = [c for c in df.columns if 'transmit' in c]

n_total      = len(df)
n_normal     = int((df['label_trace']==0).sum())
n_bottleneck = int((df['label_trace']==1).sum())
bn_rate      = n_bottleneck / n_total
normal_df    = df[df['label_trace']==0]
bottleneck_df= df[df['label_trace']==1]
y            = df['label_trace'].values

print(f"\nLoaded: {FILE_NAME}")
print(f"  {n_total:,} traces | Normal: {n_normal:,} | "
      f"BN: {n_bottleneck:,} | Rate: {bn_rate*100:.1f}%")

MIN_TRACES   = 500
BOTH_CLASSES = (n_normal > 0 and n_bottleneck > 0)
ENOUGH_DATA  = (n_total >= MIN_TRACES)

if not BOTH_CLASSES:
    status = 'BASELINE_ONLY' if n_bottleneck == 0 else 'BOTTLENECK_ONLY'
elif not ENOUGH_DATA:
    status = 'TOO_SMALL'
else:
    status = 'OK'

if status != 'OK':
    print(f"\n⚠ Skipping modeling — status: {status}")
    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    n_lat = normal_df['0_latency']/1000 if n_normal > 0 else None
    b_lat = bottleneck_df['0_latency']/1000 if n_bottleneck > 0 else None
    print(f"  Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    if n_lat is not None:
        print(f"  Normal p50: {n_lat.median():.1f}ms")
    if b_lat is not None:
        print(f"  BN p50: {b_lat.median():.1f}ms")
    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4),
        'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  Saved: {results_path}")
    os.remove(local_csv)
    print(f"  Local file deleted.")
    print(f"\n{'='*50}")
    print(f"⚠ {FILE_ID} — {status}")
    print(f"{'='*50}")

else:
    n_lat = normal_df['0_latency'] / 1000
    b_lat = bottleneck_df['0_latency'] / 1000

    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    never_set   = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates==0].index)
    df['n_bn_rpcs'] = df[rpc_cols].sum(axis=1)
    multi = df[df['label_trace']==1]['n_bn_rpcs']

    res_ratios = {
        'cpu':    bottleneck_df[cpu_cols].mean().mean() /
                  normal_df[cpu_cols].mean().mean(),
        'memory': bottleneck_df[mem_cols].mean().mean() /
                  normal_df[mem_cols].mean().mean(),
        'net_rx': bottleneck_df[net_rx_cols].mean().mean() /
                  normal_df[net_rx_cols].mean().mean(),
        'net_tx': bottleneck_df[net_tx_cols].mean().mean() /
                  normal_df[net_tx_cols].mean().mean(),
    }

    print(f"\n[L2] Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    print(f"     Never flagged ({len(never_set)}): "
          f"{sorted(never_set, key=int)}")
    print(f"     Simultaneous RPCs: always {int(multi.mode()[0])} "
          f"| Rigid: {multi.nunique()==1}")
    print(f"     Entry p50 — Normal: {n_lat.median():.1f}ms  "
          f"BN: {b_lat.median():.1f}ms  "
          f"Ratio: {b_lat.median()/n_lat.median():.2f}x")
    print(f"     Resources: CPU={res_ratios['cpu']:.4f}x  "
          f"Mem={res_ratios['memory']:.4f}x  "
          f"NetRX={res_ratios['net_rx']:.4f}x")

    node_ratios = {}
    for col in latency_cols:
        node = col.replace('_latency','')
        nv   = normal_df[col].mean() / 1000
        bv   = bottleneck_df[col].mean() / 1000
        node_ratios[node] = {
            'normal_ms': round(nv,3), 'bn_ms': round(bv,3),
            'abs_diff':  round(bv-nv,3),
            'ratio':     round(bv/nv,4) if nv>0 else 0,
            'flagged':   node in flagged_set
        }

    lat_df = df[latency_cols].copy()
    lat_df.columns = [c.replace('_latency','') for c in latency_cols]
    corr   = lat_df.corr()
    fl     = [c for c in lat_df.columns if c in flagged_set]
    unfl   = [c for c in lat_df.columns if c not in flagged_set]
    ff     = corr.loc[fl, fl].values.copy() if len(fl)>1 else np.array([[np.nan]])
    np.fill_diagonal(ff, np.nan)
    fu     = corr.loc[fl, unfl].values if (len(fl)>0 and len(unfl)>0) else np.array([[0]])

    X_lat = df[latency_cols].values
    X_res = df[cpu_cols + mem_cols + net_rx_cols + net_tx_cols].values

    feature_aucs = {}
    for name, X in [('Latency only', X_lat),
                    ('Resource only', X_res)]:
        sc = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            X, y, cv=5, scoring='roc_auc')
        feature_aucs[name] = {'mean': round(sc.mean(),4),
                              'std':  round(sc.std(),4)}

    print(f"\n[L3] Latency-only AUC: "
          f"{feature_aucs['Latency only']['mean']:.4f}  "
          f"Resource-only AUC: "
          f"{feature_aucs['Resource only']['mean']:.4f}")
    print(f"     FF corr: {np.nanmean(ff):.4f}  "
          f"FU corr: {np.nanmean(fu):.4f}")

    scaler   = StandardScaler()
    X_lat_sc = scaler.fit_transform(X_lat)
    models   = {
        'Logistic Regression': (LogisticRegression(max_iter=1000,
                                 random_state=42), X_lat_sc),
        'Random Forest':       (RandomForestClassifier(n_estimators=200,
                                 random_state=42, n_jobs=-1), X_lat),
        'Gradient Boosting':   (GradientBoostingClassifier(n_estimators=100,
                                 random_state=42), X_lat),  # fixed
    }
    model_results = {}
    for name, (model, X) in models.items():
        auc_s = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
        f1_s  = cross_val_score(model, X, y, cv=5, scoring='f1')
        model_results[name] = {'auc': round(auc_s.mean(),4),
                               'f1':  round(f1_s.mean(),4),
                               'std': round(auc_s.std(),4)}

    baseline_auc = cross_val_score(
        RandomForestClassifier(n_estimators=100,
                               random_state=42, n_jobs=-1),
        X_lat, y, cv=5, scoring='roc_auc').mean()

    ablation = {}
    for i, col in enumerate(latency_cols):
        node = col.replace('_latency','')
        sc   = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            np.delete(X_lat, i, axis=1), y, cv=3,
            scoring='roc_auc').mean()
        ablation[node] = round(sc,4)

    drops   = {n: round(baseline_auc-sc,4) for n,sc in ablation.items()}
    drops_s = sorted(drops.items(), key=lambda x: x[1], reverse=True)

    print(f"\n[L4] RF AUC: {model_results['Random Forest']['auc']:.4f}  "
          f"F1: {model_results['Random Forest']['f1']:.4f}")
    print(f"     LR AUC: "
          f"{model_results['Logistic Regression']['auc']:.4f}  "
          f"GB AUC: {model_results['Gradient Boosting']['auc']:.4f}")
    print(f"     Top ablation: {drops_s[:3]}")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Analysis: {FILE_NAME}\n'
                 f'{WORKFLOW} | {BN_TYPE} | {LOAD_RPS} RPS | Run {RUN_NUM}',
                 fontsize=12, fontweight='bold')

    ax = axes[0,0]
    ax.bar(['Normal','Bottleneck'], [n_normal, n_bottleneck],
           color=['#2196F3','#F44336'], edgecolor='white', width=0.5)
    ax.set_title('Class balance', fontweight='bold')
    ax.set_ylim(0, max(n_normal, n_bottleneck)*1.25)
    for i, val in enumerate([n_normal, n_bottleneck]):
        ax.text(i, val + max(n_normal,n_bottleneck)*0.02,
                f'{val:,}\n({val/n_total*100:.1f}%)',
                ha='center', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[0,1]
    ax.bar(range(30), [1]*30,
           color=['#F44336' if str(n) in flagged_set
                  else '#E0E0E0' for n in range(30)],
           edgecolor='white', width=0.8)
    ax.set_xticks(range(30))
    ax.set_xticklabels([str(n) for n in range(30)], fontsize=6, rotation=90)
    ax.set_yticks([])
    ax.set_title(f'Flagged nodes ({len(flagged_set)}/30)', fontweight='bold')
    ax.spines[['top','right','left']].set_visible(False)

    ax = axes[0,2]
    nodes_s  = sorted(node_ratios.keys(), key=int)
    ratios_v = [node_ratios[n]['ratio'] for n in nodes_s]
    ax.bar(range(30), ratios_v,
           color=['#F44336' if node_ratios[n]['flagged']
                  else '#90CAF9' for n in nodes_s],
           edgecolor='white', width=0.7)
    ax.axhline(1.0, color='gray', linestyle='--', lw=1, alpha=0.7)
    ax.set_xticks(range(30))
    ax.set_xticklabels(nodes_s, fontsize=6, rotation=90)
    ax.set_title('Per-node latency ratio', fontweight='bold')
    ax.set_ylabel('BN / Normal')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,0]
    rv = list(res_ratios.values())
    ax.barh(list(res_ratios.keys()), rv,
            color=['#EF5350' if v>1.1 else '#FFA726'
                   if v>1.03 else '#66BB6A' for v in rv],
            edgecolor='white')
    ax.axvline(1.0, color='gray', linestyle='--', lw=1)
    ax.set_title('Resource ratios', fontweight='bold')
    for i, v in enumerate(rv):
        ax.text(v+0.001, i, f'{v:.4f}x', va='center', fontsize=9)
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,1]
    fa_vals = [feature_aucs[k]['mean'] for k in feature_aucs]
    ax.bar(range(2), fa_vals, color=['#4CAF50','#FF9800'],
           edgecolor='white', width=0.4)
    ax.set_xticks(range(2))
    ax.set_xticklabels(['Latency\nonly','Resource\nonly'], fontsize=9)
    ax.set_ylim(0.5, 1.05)
    ax.set_title('AUC by feature set (RF)', fontweight='bold')
    for i, v in enumerate(fa_vals):
        ax.text(i, v+0.008, f'{v:.4f}', ha='center',
                fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,2]
    ax.barh(range(8), [v for _,v in drops_s[:8]][::-1],
            color=['#D32F2F' if v>0.008 else '#FF9800'
                   if v>0.004 else '#FFC107'
                   for _,v in drops_s[:8]][::-1],
            edgecolor='white')
    ax.set_yticks(range(8))
    ax.set_yticklabels([f'node {n}' for n,_ in drops_s[:8]][::-1], fontsize=8)
    ax.set_title(f'Node ablation (base={baseline_auc:.4f})', fontweight='bold')
    ax.set_xlabel('AUC drop')
    ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    chart_path = os.path.join(CHARTS_DIR, f'{FILE_ID}_analysis.png')
    plt.savefig(chart_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Chart saved: {chart_path}")

    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4), 'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
        'never_flagged': sorted(never_set, key=int),
        'n_flagged': len(flagged_set),
        'always_same_subgraph': bool(multi.nunique()==1),
        'simultaneous_rpcs': int(multi.mode()[0]),
        'entry_latency': {
            'normal_p50_ms': round(n_lat.median(),2),
            'bn_p50_ms':     round(b_lat.median(),2),
            'ratio_p50':     round(b_lat.median()/n_lat.median(),4),
            'ratio_mean':    round(b_lat.mean()/n_lat.mean(),4),
        },
        'resource_ratios': {k: round(v,4) for k,v in res_ratios.items()},
        'feature_aucs':   feature_aucs,
        'model_results':  model_results,
        'baseline_auc':   round(baseline_auc,4),
        'ablation_top8':  {n: v for n,v in drops_s[:8]},
        'node_ratios':    node_ratios,
        'ff_corr': round(float(np.nanmean(ff)),4),
        'fu_corr': round(float(np.nanmean(fu)),4),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Results saved: {results_path}")
    os.remove(local_csv)
    print(f"Local file deleted.")
    print(f"\n{'='*50}")
    print(f"✓ {FILE_ID} COMPLETE")
    print(f"{'='*50}")

## Batch 5a Results: CPU+Memory combined — 800 RPS — sep29 (runs 0–8)
*Files: cpu_mem_sep29_10min_800_[0-8]_graph_1.csv | 8 valid + 1 wrong file*
*Note: run 8 accidentally downloaded mem_sep25_8 — must be rerun*

### CRITICAL: Run 3 reveals a FIFTH pattern — 19 nodes

Run 3 flags **19 nodes** — the largest subgraph in the entire dataset:
**{0,1,2,4,5,7,8,11,12,13,14,18,19,20,21,22,26,27,28}**

This is Pattern A (15 nodes) + Pattern C entry nodes {0,1,2,22} merged
into a single combined subgraph. It is Pattern D (11) plus the 8 middle
nodes {4,5,7,8,11,12,18,19} that Pattern D was missing.

In other words: **Pattern E = Pattern A ∪ Pattern C = all 19 nodes**

### Complete pattern taxonomy — now 5 patterns

| Pattern | Name | Nodes | Count | Bottleneck type |
|---------|------|-------|-------|----------------|
| A | Wide storage | {4,5,7,8,11,12,13,14,18,19,20,21,26,27,28} | 15 | CPU only |
| B | Core storage | {4,5,13,14,20,21,26,27,28} | 9 | CPU only |
| C | Entry layer | {0,1,2,22} | 4 | CPU only |
| D | Entry+Core hybrid | {0,1,2,13,14,20,21,22,26,27,28} | 11 | CPU + Memory |
| **E** | **Full system** | **{0,1,2,4,5,7,8,11,12,13,14,18,19,20,21,22,26,27,28}** | **19** | **CPU+Memory** |

Pattern E = Pattern A ∪ Pattern C. The combined CPU+Memory stress
activates both the storage chain AND the entry layer simultaneously,
with the full middle chain {4,5,7,8,11,12,18,19} also activated.
Only 11 nodes remain unaffected: {3,6,9,10,15,16,17,23,24,25,29}.

### Most runs still show Pattern D — run 3 is the exception

| Run | Flagged nodes | Count | Pattern | RF AUC |
|-----|--------------|-------|---------|--------|
| 0 | {0,1,2,13,14,20,21,22,26,27,28} | 11 | D | 0.768 |
| 1 | {0,1,2,13,14,20,21,22,26,27,28} | 11 | D | 0.734 |
| 2 | {0,1,2,13,14,20,21,22,26,27,28} | 11 | D | 0.845 |
| **3** | **{0,1,2,4,5,7,8,11,12,13,14,18,19,20,21,22,26,27,28}** | **19** | **E** | **0.927** |
| 4 | {0,1,2,13,14,20,21,22,26,27,28} | 11 | D | 0.955 |
| 5 | {0,1,2,13,14,20,21,22,26,27,28} | 11 | D | 0.900 |
| 6 | {0,1,2,13,14,20,21,22,26,27,28} | 11 | D | 0.787 |
| 7 | {0,1,2,13,14,20,21,22,26,27,28} | 11 | D | 0.661 |

Pattern D dominates (7 of 8 valid runs). Run 3 is the only Pattern E
appearance so far — but it is perfectly rigid (all 19 flagged in 100%
of bottleneck traces). This suggests Pattern E requires a specific
injection intensity that was only achieved in run 3.

### Run 0, 1, 6, 7: elevated normal baseline — system pre-stressed
Normal p50 in these runs: 50–61ms vs 20–28ms in runs 2,3,4,5.
The system was already under load before injection, making the
bottleneck signal weaker and AUC lower (0.661–0.787 vs 0.845–0.955).
This explains why run 7 has the lowest AUC (0.661) despite the same
subgraph pattern.

### Run 3: Pattern E signal characteristics
- Entry p50 ratio: **7.32×** — strongest in the cpu_mem group
- RF AUC: 0.927 — highest in the batch
- LR AUC: 0.901 — unusually close to RF (linear boundary almost sufficient)
- FU correlation: 0.253 — lower than other cpu_mem runs (more localized)
- All ablation values negative — signal is distributed, no single dominant node

### Theory of Constraints — Pattern E
Pattern E represents a **full system constraint**: both the entry layer
and the entire storage chain are simultaneously bottlenecked. This is the
most severe constraint state — it leaves only 11 peripheral/silent nodes
unaffected. In ToC terms: two simultaneous constraints in independent
parallel paths of the call graph.

---
## Action required before continuing
Re-run Cell 5 with:
FILE_NAME = 'cpu_mem_sep29_10min_800_8_graph_1.csv'
RUN_NUM = 8

### Run 8 addendum
Pattern D confirmed — 11 nodes {0,1,2,13,14,20,21,22,26,27,28}.
Normal p50 = **77.2ms** — the highest baseline in the entire dataset.
The system was severely pre-stressed before injection, compressing
the bottleneck signal to only 2.89× and driving AUC to 0.715.
Resource-only AUC = 0.995 — the only case where resources outperform
latency. When the system is already saturated, resource signals become
more discriminative than latency signals. This is a notable exception
to the general rule established across the dataset.

### sep29 complete summary
- 8 of 9 runs: Pattern D (11 nodes)
- 1 of 9 runs: Pattern E (19 nodes) — run 3 only
- AUC range: 0.661 – 0.955
- Key insight: elevated baseline (pre-stress) is the primary driver
  of low AUC, more than bottleneck type or injection intensity

---
## Batch 5b: CPU+Memory combined — oct2 (runs 0–29)
*Files: cpu_mem_oct2_10min_800_[0-29]_graph_1.csv*
*Question: Does Pattern E appear more consistently in the oct2 group?*

## Batch 5b Results: CPU+Memory combined — 800 RPS — oct2 (runs 0–29)
*Files: cpu_mem_oct2_10min_800_[0-29]_graph_1.csv | 30 files | ~466,000 traces*

### Pattern D dominates — Pattern E appears in run 1 only

| Pattern | Files | Percentage |
|---------|-------|-----------|
| D — 11 nodes {0,1,2,13,14,20,21,22,26,27,28} | 29 of 30 | 96.7% |
| E — 19 nodes {0,1,2,4,5,7,8,11,12,13,14,18,19,20,21,22,26,27,28} | 1 of 30 | 3.3% |

Pattern E appeared only in run 1. Combined with sep29 run 3, Pattern E
has now appeared in exactly 2 of 39 CPU+Memory files (5.1%).
It is a real but rare configuration — likely requiring a specific
injection intensity threshold to activate the middle chain.

### Quantitative summary — oct2 full batch

| Metric | Range | Note |
|--------|-------|------|
| BN rate | 38.0% – 57.5% | Most balanced batch in dataset |
| Normal p50 baseline | 26–59ms | Pre-stress elevated in many runs |
| Entry p50 ratio | 2.83× – 8.43× | Wide variation |
| RF AUC | **0.689 – 0.909** | Large spread |
| LR AUC | 0.630 – 0.867 | LR competitive here |
| Memory ratio | 0.922× – 1.099× | Never reliably above 1.1× |

### Elevated baseline is the primary AUC predictor

The clearest pattern in oct2: normal p50 latency before injection
strongly predicts AUC. When the system starts stressed, detection fails.

| Normal p50 range | Mean RF AUC |
|-----------------|-------------|
| 26–35ms (fresh system) | ~0.855 |
| 36–45ms (mild pre-stress) | ~0.800 |
| 46–60ms (heavy pre-stress) | ~0.725 |

This is the strongest evidence yet that **baseline system health
matters more than bottleneck type for detection difficulty**.

### Resource signals show a striking pattern in oct2
Resource-only AUC is consistently very high: 0.821 – 0.993.
Yet latency-only AUC is 0.689 – 0.909.
In 26 of 30 files: Resource AUC > Latency AUC.

This is the **opposite** of CPU-only bottlenecks where latency
dominated. Combined CPU+Memory stress produces resource signatures
strong enough to rival latency signals. However, resource signals
are still unreliable for identifying *which* nodes are constrained
— they only detect *that* a bottleneck exists.

### Pattern E characteristics — run 1
- 19 nodes: {0,1,2,4,5,7,8,11,12,13,14,18,19,20,21,22,26,27,28}
- Entry ratio: 4.49× | RF AUC: 0.700
- Paradox: despite more nodes flagged, AUC is LOWER than most D runs
- All ablation values negative — no single node is informative
- The 19-node signal is distributed so widely that no individual
  node stands out, making the classifier work harder

### Complete CPU+Memory group summary (sep29 + oct2 = 39 files)

| Metric | Full range |
|--------|-----------|
| Pattern D | 37 of 39 files (94.9%) |
| Pattern E | 2 of 39 files (5.1%) |
| RF AUC | 0.661 – 0.955 |
| Entry p50 ratio | 2.22× – 8.43× |
| Memory ratio | 0.871× – 1.099× — never a reliable signal |
| FU correlation | 0.32 – 0.41 |

### Theory of Constraints — CPU+Memory combined
Adding memory stress on top of CPU stress does NOT create a new
constraint location. The system constraint remains the same
11-node subgraph (Pattern D) in 95% of cases. The constraint is
structurally determined by the call graph, not by how many resource
types are stressed simultaneously. Only at high injection intensity
does the middle chain {4,5,7,8,11,12,18,19} activate (Pattern E),
suggesting these nodes have a higher stress threshold before
becoming part of the constraint path.

---
## CPU+Memory Complete — Moving to Network Bottleneck

### Updated pattern frequency across all 167 analyzed files

| Pattern | Total files | Bottleneck types |
|---------|------------|-----------------|
| D (11 nodes) | ~92 files | CPU sept/sept20, Memory, CPU+Memory |
| A (15 nodes) | ~31 files | CPU aug12/18/sept4 |
| C (4 nodes)  | ~17 files | CPU july |
| B (9 nodes)  | ~10 files | CPU aug30 |
| E (19 nodes) | ~2 files  | CPU+Memory only |

### The central finding crystallizes
Across 167 files and 3 bottleneck types, the constraint location
is determined by injection point, not resource type:
- Storage-layer injection → Patterns A, B, D
- Entry-layer injection → Pattern C (or D when both layers hit)
- High-intensity combined injection → Pattern E (rare)

The irreducible core {13,14,20,21,26,27,28} appears in A, B, D, E —
confirming it as the structural constraint of the compose workflow
under any stress configuration targeting the storage layer.

---
## Batch 6: Network Bottleneck — oct4 (runs 0–29)
*Files: net_oct4_10min_800_[0-29]_graph_1.csv*
*BN_TYPE = 'Network throttle' | LOAD_RPS = 800*
*Key question: Does network throttling activate a completely different
subgraph, or does it converge to the same structural patterns?*
*Prediction: likely Pattern D or a new Pattern F — network delays
propagate differently than compute resource exhaustion.*

In [ ]:
# ============================================================
# CELL 5 — Corrected (re-run for batch 6)
# ============================================================

FILE_NAME   = 'cpu_july28_800_0_graph_1.csv'
BN_TYPE     = 'CPU stress'
WORKFLOW    = 'Compose'
LOAD_RPS    = 800
RUN_NUM     = 0

KAGGLE_PATH = (
    'processed_dataset/compose/multi-modal-data-separate/'
    + FILE_NAME
)
FILE_ID = (FILE_NAME
           .replace('_graph_1.csv','')
           .replace('_25min_repeat','')
           .replace('_25min_rerun','')
           .replace('_25min','')
           .replace('_30min',''))

LOCAL_DIR = '/content/ms_data'
local_csv = os.path.join(LOCAL_DIR, FILE_NAME)
os.makedirs(LOCAL_DIR, exist_ok=True)

if not os.path.exists(local_csv):
    print(f"Downloading {FILE_NAME}...")
    result = subprocess.run([
        'kaggle', 'datasets', 'download',
        'gagansomashekar/microservices-bottleneck-detection-dataset',
        '--path', LOCAL_DIR, '--file', KAGGLE_PATH
    ], capture_output=True, text=True)
    for f in os.listdir(LOCAL_DIR):
        if f.endswith('.zip'):
            with zipfile.ZipFile(os.path.join(LOCAL_DIR, f)) as z:
                z.extractall(LOCAL_DIR)
            os.remove(os.path.join(LOCAL_DIR, f))
    print("Downloaded and unzipped.")
else:
    print(f"Already exists: {FILE_NAME}")

df = pd.read_csv(local_csv)

latency_cols = [c for c in df.columns if c.endswith('_latency')]
rpc_cols     = [c for c in df.columns if c.endswith('_label_RPC')]
cpu_cols     = [c for c in df.columns if 'cpu' in c and 'label' not in c]
mem_cols     = [c for c in df.columns if 'memory' in c]
net_rx_cols  = [c for c in df.columns if 'receive' in c]
net_tx_cols  = [c for c in df.columns if 'transmit' in c]

n_total      = len(df)
n_normal     = int((df['label_trace']==0).sum())
n_bottleneck = int((df['label_trace']==1).sum())
bn_rate      = n_bottleneck / n_total
normal_df    = df[df['label_trace']==0]
bottleneck_df= df[df['label_trace']==1]
y            = df['label_trace'].values

print(f"\nLoaded: {FILE_NAME}")
print(f"  {n_total:,} traces | Normal: {n_normal:,} | "
      f"BN: {n_bottleneck:,} | Rate: {bn_rate*100:.1f}%")

MIN_TRACES   = 500
BOTH_CLASSES = (n_normal > 0 and n_bottleneck > 0)
ENOUGH_DATA  = (n_total >= MIN_TRACES)

if not BOTH_CLASSES:
    status = 'BASELINE_ONLY' if n_bottleneck == 0 else 'BOTTLENECK_ONLY'
elif not ENOUGH_DATA:
    status = 'TOO_SMALL'
else:
    status = 'OK'

if status != 'OK':
    print(f"\n⚠ Skipping modeling — status: {status}")
    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    n_lat = normal_df['0_latency']/1000 if n_normal > 0 else None
    b_lat = bottleneck_df['0_latency']/1000 if n_bottleneck > 0 else None
    print(f"  Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    if n_lat is not None:
        print(f"  Normal p50: {n_lat.median():.1f}ms")
    if b_lat is not None:
        print(f"  BN p50: {b_lat.median():.1f}ms")
    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4),
        'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  Saved: {results_path}")
    os.remove(local_csv)
    print(f"  Local file deleted.")
    print(f"\n{'='*50}")
    print(f"⚠ {FILE_ID} — {status}")
    print(f"{'='*50}")

else:
    n_lat = normal_df['0_latency'] / 1000
    b_lat = bottleneck_df['0_latency'] / 1000

    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    never_set   = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates==0].index)
    df['n_bn_rpcs'] = df[rpc_cols].sum(axis=1)
    multi = df[df['label_trace']==1]['n_bn_rpcs']

    res_ratios = {
        'cpu':    bottleneck_df[cpu_cols].mean().mean() /
                  normal_df[cpu_cols].mean().mean(),
        'memory': bottleneck_df[mem_cols].mean().mean() /
                  normal_df[mem_cols].mean().mean(),
        'net_rx': bottleneck_df[net_rx_cols].mean().mean() /
                  normal_df[net_rx_cols].mean().mean(),
        'net_tx': bottleneck_df[net_tx_cols].mean().mean() /
                  normal_df[net_tx_cols].mean().mean(),
    }

    print(f"\n[L2] Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    print(f"     Never flagged ({len(never_set)}): "
          f"{sorted(never_set, key=int)}")
    print(f"     Simultaneous RPCs: always {int(multi.mode()[0])} "
          f"| Rigid: {multi.nunique()==1}")
    print(f"     Entry p50 — Normal: {n_lat.median():.1f}ms  "
          f"BN: {b_lat.median():.1f}ms  "
          f"Ratio: {b_lat.median()/n_lat.median():.2f}x")
    print(f"     Resources: CPU={res_ratios['cpu']:.4f}x  "
          f"Mem={res_ratios['memory']:.4f}x  "
          f"NetRX={res_ratios['net_rx']:.4f}x")

    node_ratios = {}
    for col in latency_cols:
        node = col.replace('_latency','')
        nv   = normal_df[col].mean() / 1000
        bv   = bottleneck_df[col].mean() / 1000
        node_ratios[node] = {
            'normal_ms': round(nv,3), 'bn_ms': round(bv,3),
            'abs_diff':  round(bv-nv,3),
            'ratio':     round(bv/nv,4) if nv>0 else 0,
            'flagged':   node in flagged_set
        }

    lat_df = df[latency_cols].copy()
    lat_df.columns = [c.replace('_latency','') for c in latency_cols]
    corr   = lat_df.corr()
    fl     = [c for c in lat_df.columns if c in flagged_set]
    unfl   = [c for c in lat_df.columns if c not in flagged_set]
    ff     = corr.loc[fl, fl].values.copy() if len(fl)>1 else np.array([[np.nan]])
    np.fill_diagonal(ff, np.nan)
    fu     = corr.loc[fl, unfl].values if (len(fl)>0 and len(unfl)>0) else np.array([[0]])

    X_lat = df[latency_cols].values
    X_res = df[cpu_cols + mem_cols + net_rx_cols + net_tx_cols].values

    feature_aucs = {}
    for name, X in [('Latency only', X_lat),
                    ('Resource only', X_res)]:
        sc = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            X, y, cv=5, scoring='roc_auc')
        feature_aucs[name] = {'mean': round(sc.mean(),4),
                              'std':  round(sc.std(),4)}

    print(f"\n[L3] Latency-only AUC: "
          f"{feature_aucs['Latency only']['mean']:.4f}  "
          f"Resource-only AUC: "
          f"{feature_aucs['Resource only']['mean']:.4f}")
    print(f"     FF corr: {np.nanmean(ff):.4f}  "
          f"FU corr: {np.nanmean(fu):.4f}")

    scaler   = StandardScaler()
    X_lat_sc = scaler.fit_transform(X_lat)
    models   = {
        'Logistic Regression': (LogisticRegression(max_iter=1000,
                                 random_state=42), X_lat_sc),
        'Random Forest':       (RandomForestClassifier(n_estimators=200,
                                 random_state=42, n_jobs=-1), X_lat),
        'Gradient Boosting':   (GradientBoostingClassifier(n_estimators=100,
                                 random_state=42), X_lat),  # fixed
    }
    model_results = {}
    for name, (model, X) in models.items():
        auc_s = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
        f1_s  = cross_val_score(model, X, y, cv=5, scoring='f1')
        model_results[name] = {'auc': round(auc_s.mean(),4),
                               'f1':  round(f1_s.mean(),4),
                               'std': round(auc_s.std(),4)}

    baseline_auc = cross_val_score(
        RandomForestClassifier(n_estimators=100,
                               random_state=42, n_jobs=-1),
        X_lat, y, cv=5, scoring='roc_auc').mean()

    ablation = {}
    for i, col in enumerate(latency_cols):
        node = col.replace('_latency','')
        sc   = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            np.delete(X_lat, i, axis=1), y, cv=3,
            scoring='roc_auc').mean()
        ablation[node] = round(sc,4)

    drops   = {n: round(baseline_auc-sc,4) for n,sc in ablation.items()}
    drops_s = sorted(drops.items(), key=lambda x: x[1], reverse=True)

    print(f"\n[L4] RF AUC: {model_results['Random Forest']['auc']:.4f}  "
          f"F1: {model_results['Random Forest']['f1']:.4f}")
    print(f"     LR AUC: "
          f"{model_results['Logistic Regression']['auc']:.4f}  "
          f"GB AUC: {model_results['Gradient Boosting']['auc']:.4f}")
    print(f"     Top ablation: {drops_s[:3]}")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Analysis: {FILE_NAME}\n'
                 f'{WORKFLOW} | {BN_TYPE} | {LOAD_RPS} RPS | Run {RUN_NUM}',
                 fontsize=12, fontweight='bold')

    ax = axes[0,0]
    ax.bar(['Normal','Bottleneck'], [n_normal, n_bottleneck],
           color=['#2196F3','#F44336'], edgecolor='white', width=0.5)
    ax.set_title('Class balance', fontweight='bold')
    ax.set_ylim(0, max(n_normal, n_bottleneck)*1.25)
    for i, val in enumerate([n_normal, n_bottleneck]):
        ax.text(i, val + max(n_normal,n_bottleneck)*0.02,
                f'{val:,}\n({val/n_total*100:.1f}%)',
                ha='center', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[0,1]
    ax.bar(range(30), [1]*30,
           color=['#F44336' if str(n) in flagged_set
                  else '#E0E0E0' for n in range(30)],
           edgecolor='white', width=0.8)
    ax.set_xticks(range(30))
    ax.set_xticklabels([str(n) for n in range(30)], fontsize=6, rotation=90)
    ax.set_yticks([])
    ax.set_title(f'Flagged nodes ({len(flagged_set)}/30)', fontweight='bold')
    ax.spines[['top','right','left']].set_visible(False)

    ax = axes[0,2]
    nodes_s  = sorted(node_ratios.keys(), key=int)
    ratios_v = [node_ratios[n]['ratio'] for n in nodes_s]
    ax.bar(range(30), ratios_v,
           color=['#F44336' if node_ratios[n]['flagged']
                  else '#90CAF9' for n in nodes_s],
           edgecolor='white', width=0.7)
    ax.axhline(1.0, color='gray', linestyle='--', lw=1, alpha=0.7)
    ax.set_xticks(range(30))
    ax.set_xticklabels(nodes_s, fontsize=6, rotation=90)
    ax.set_title('Per-node latency ratio', fontweight='bold')
    ax.set_ylabel('BN / Normal')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,0]
    rv = list(res_ratios.values())
    ax.barh(list(res_ratios.keys()), rv,
            color=['#EF5350' if v>1.1 else '#FFA726'
                   if v>1.03 else '#66BB6A' for v in rv],
            edgecolor='white')
    ax.axvline(1.0, color='gray', linestyle='--', lw=1)
    ax.set_title('Resource ratios', fontweight='bold')
    for i, v in enumerate(rv):
        ax.text(v+0.001, i, f'{v:.4f}x', va='center', fontsize=9)
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,1]
    fa_vals = [feature_aucs[k]['mean'] for k in feature_aucs]
    ax.bar(range(2), fa_vals, color=['#4CAF50','#FF9800'],
           edgecolor='white', width=0.4)
    ax.set_xticks(range(2))
    ax.set_xticklabels(['Latency\nonly','Resource\nonly'], fontsize=9)
    ax.set_ylim(0.5, 1.05)
    ax.set_title('AUC by feature set (RF)', fontweight='bold')
    for i, v in enumerate(fa_vals):
        ax.text(i, v+0.008, f'{v:.4f}', ha='center',
                fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,2]
    ax.barh(range(8), [v for _,v in drops_s[:8]][::-1],
            color=['#D32F2F' if v>0.008 else '#FF9800'
                   if v>0.004 else '#FFC107'
                   for _,v in drops_s[:8]][::-1],
            edgecolor='white')
    ax.set_yticks(range(8))
    ax.set_yticklabels([f'node {n}' for n,_ in drops_s[:8]][::-1], fontsize=8)
    ax.set_title(f'Node ablation (base={baseline_auc:.4f})', fontweight='bold')
    ax.set_xlabel('AUC drop')
    ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    chart_path = os.path.join(CHARTS_DIR, f'{FILE_ID}_analysis.png')
    plt.savefig(chart_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Chart saved: {chart_path}")

    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4), 'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
        'never_flagged': sorted(never_set, key=int),
        'n_flagged': len(flagged_set),
        'always_same_subgraph': bool(multi.nunique()==1),
        'simultaneous_rpcs': int(multi.mode()[0]),
        'entry_latency': {
            'normal_p50_ms': round(n_lat.median(),2),
            'bn_p50_ms':     round(b_lat.median(),2),
            'ratio_p50':     round(b_lat.median()/n_lat.median(),4),
            'ratio_mean':    round(b_lat.mean()/n_lat.mean(),4),
        },
        'resource_ratios': {k: round(v,4) for k,v in res_ratios.items()},
        'feature_aucs':   feature_aucs,
        'model_results':  model_results,
        'baseline_auc':   round(baseline_auc,4),
        'ablation_top8':  {n: v for n,v in drops_s[:8]},
        'node_ratios':    node_ratios,
        'ff_corr': round(float(np.nanmean(ff)),4),
        'fu_corr': round(float(np.nanmean(fu)),4),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Results saved: {results_path}")
    os.remove(local_csv)
    print(f"Local file deleted.")
    print(f"\n{'='*50}")
    print(f"✓ {FILE_ID} COMPLETE")
    print(f"{'='*50}")

## Batch 6: Network Bottleneck — 800 RPS — oct4 (runs 0–29)
*Files: net_oct4_10min_800_[0-29]_graph_1.csv | 30 files | ~484,000 traces*

### MAJOR FINDING: Network throttling produces Pattern F — a sixth pattern

All 30 files flag exactly the same 11 nodes — but NOT the same 11 as Pattern D.

**Network pattern: {0, 1, 2, 4, 5, 7, 8, 11, 12, 18, 19}**
**Pattern D was: {0, 1, 2, 13, 14, 20, 21, 22, 26, 27, 28}**

These two sets share only 3 nodes {0,1,2} and are otherwise completely different.
This is Pattern F — the sixth and final pattern in the taxonomy.

### Pattern F vs Pattern D — direct comparison

| Property | Pattern D | Pattern F |
|----------|-----------|-----------|
| Flagged nodes | {0,1,2,13,14,20,21,22,26,27,28} | {0,1,2,4,5,7,8,11,12,18,19} |
| Layer affected | Entry + Storage core | Entry + Middle chain |
| Storage core {13,14,20,21,26,27,28} | YES — all 7 present | **ABSENT — 0 of 7** |
| Middle chain {4,5,7,8,11,12,18,19} | ABSENT | **YES — all 8 present** |
| Entry layer {0,1,2} | YES | YES |
| Node 22 | YES | **NO** |
| Bottleneck type | CPU+Memory | **Network only** |

Network throttling hits a completely different layer of the architecture.
It affects the message-passing and RPC-calling middle chain but leaves the
storage core completely untouched. This is architecturally logical:
network delays slow down inter-service communication (middle chain RPCs)
but do not directly stress the storage services themselves.

### The irreducible core {13,14,20,21,26,27,28} is ABSENT in Pattern F
This is the most important structural finding of the entire analysis.
The 7 storage services that appeared in every CPU and memory bottleneck
do NOT appear under network throttling. Network stress does not reach
the storage layer — it gets absorbed by the middle chain first.

This definitively proves: **the constraint location is determined by
the propagation path of the stress through the call graph, not by
the resource type or load level.**

### Quantitative summary — network batch

| Metric | Range | Note |
|--------|-------|------|
| BN rate | 22.1% – 69.0% | Wide — runs 26-29 have very low BN rate |
| Entry p50 ratio | **1.09× – 4.79×** | Extreme variation |
| RF AUC | **0.593 – 0.883** | Widest AUC range in entire dataset |
| LR AUC | 0.539 – 0.875 | LR struggles here |
| Resource NetRX | 1.10× – 1.74× | Elevated but not reliable |

### Two distinct sub-groups within the network batch

**High-signal group (runs 0–16): ratio 2.57–4.79×, AUC 0.777–0.883**
Network throttling is strong enough to produce clear latency separation.
Node 0 leads ablation in all 17 runs. Node 5 consistently second.

**Low-signal group (runs 17–29): ratio 1.09–2.03×, AUC 0.593–0.784**
Very weak throttling — the system barely shows stress. Run 27 reaches
AUC=0.593, the lowest in the entire 196-file dataset. F1=0.163 means
the model can barely distinguish bottleneck from normal traces.
Yet the flagged node set is STILL exactly the same 11 nodes — the
pattern is rigid even when the signal is near-zero.

### Exceptional ablation values in low-signal runs
Runs 20, 24, 25: ablation drops of 0.10–0.125 — among the largest
in the entire dataset. When the overall signal is weak, a handful of
nodes carry almost all the discriminative information. The model
becomes extremely dependent on nodes 4, 5, 8 in these cases.

### Resource signals: NetRX is relevant here — but still not sufficient
Unlike CPU/memory bottlenecks where resources were useless, network
throttling does elevate NetRX (1.10–1.74×). In many runs
Resource-only AUC > Latency-only AUC.
However resource signals still cannot identify WHICH nodes are
bottlenecked — only that a bottleneck exists somewhere.
Latency remains the only signal that pinpoints the constraint path.

### FF correlation is highest in the dataset: 0.26–0.63
The 11 flagged nodes in Pattern F are more internally correlated than
any previous pattern. This makes sense — the middle chain nodes are
tightly coupled sequential RPCs. When network is throttled, they all
slow down together and their latencies move in near-perfect lockstep.

---
## COMPLETE PATTERN TAXONOMY — ALL 6 PATTERNS CONFIRMED

| Pattern | Name | Nodes | Set | Bottleneck type | AUC range |
|---------|------|-------|-----|----------------|-----------|
| A | Wide storage | 15 | {4,5,7,8,11,12,13,14,18,19,20,21,26,27,28} | CPU only | 0.955–0.981 |
| B | Core storage | 9 | {4,5,13,14,20,21,26,27,28} | CPU only | 0.815–0.847 |
| C | Entry layer | 4 | {0,1,2,22} | CPU only | 0.790–0.958 |
| D | Entry+Core hybrid | 11 | {0,1,2,13,14,20,21,22,26,27,28} | CPU+Memory | 0.635–0.959 |
| E | Full system | 19 | {0,1,2,4,5,7,8,11,12,13,14,18,19,20,21,22,26,27,28} | CPU+Memory (rare) | 0.700–0.927 |
| **F** | **Entry+Middle chain** | **11** | **{0,1,2,4,5,7,8,11,12,18,19}** | **Network only** | **0.593–0.883** |

### Mathematical relationships — updated

In [ ]:
# ============================================================
# CELL 5 — Corrected (re-run for batch 7)
# ============================================================

FILE_NAME   = 'cpu_july28_800_0_graph_1.csv'
BN_TYPE     = 'CPU stress'
WORKFLOW    = 'Compose'
LOAD_RPS    = 800
RUN_NUM     = 0

KAGGLE_PATH = (
    'processed_dataset/compose/multi-modal-data-separate/'
    + FILE_NAME
)
FILE_ID = (FILE_NAME
           .replace('_graph_1.csv','')
           .replace('_25min_repeat','')
           .replace('_25min_rerun','')
           .replace('_25min','')
           .replace('_30min',''))

LOCAL_DIR = '/content/ms_data'
local_csv = os.path.join(LOCAL_DIR, FILE_NAME)
os.makedirs(LOCAL_DIR, exist_ok=True)

if not os.path.exists(local_csv):
    print(f"Downloading {FILE_NAME}...")
    result = subprocess.run([
        'kaggle', 'datasets', 'download',
        'gagansomashekar/microservices-bottleneck-detection-dataset',
        '--path', LOCAL_DIR, '--file', KAGGLE_PATH
    ], capture_output=True, text=True)
    for f in os.listdir(LOCAL_DIR):
        if f.endswith('.zip'):
            with zipfile.ZipFile(os.path.join(LOCAL_DIR, f)) as z:
                z.extractall(LOCAL_DIR)
            os.remove(os.path.join(LOCAL_DIR, f))
    print("Downloaded and unzipped.")
else:
    print(f"Already exists: {FILE_NAME}")

df = pd.read_csv(local_csv)

latency_cols = [c for c in df.columns if c.endswith('_latency')]
rpc_cols     = [c for c in df.columns if c.endswith('_label_RPC')]
cpu_cols     = [c for c in df.columns if 'cpu' in c and 'label' not in c]
mem_cols     = [c for c in df.columns if 'memory' in c]
net_rx_cols  = [c for c in df.columns if 'receive' in c]
net_tx_cols  = [c for c in df.columns if 'transmit' in c]

n_total      = len(df)
n_normal     = int((df['label_trace']==0).sum())
n_bottleneck = int((df['label_trace']==1).sum())
bn_rate      = n_bottleneck / n_total
normal_df    = df[df['label_trace']==0]
bottleneck_df= df[df['label_trace']==1]
y            = df['label_trace'].values

print(f"\nLoaded: {FILE_NAME}")
print(f"  {n_total:,} traces | Normal: {n_normal:,} | "
      f"BN: {n_bottleneck:,} | Rate: {bn_rate*100:.1f}%")

MIN_TRACES   = 500
BOTH_CLASSES = (n_normal > 0 and n_bottleneck > 0)
ENOUGH_DATA  = (n_total >= MIN_TRACES)

if not BOTH_CLASSES:
    status = 'BASELINE_ONLY' if n_bottleneck == 0 else 'BOTTLENECK_ONLY'
elif not ENOUGH_DATA:
    status = 'TOO_SMALL'
else:
    status = 'OK'

if status != 'OK':
    print(f"\n⚠ Skipping modeling — status: {status}")
    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    n_lat = normal_df['0_latency']/1000 if n_normal > 0 else None
    b_lat = bottleneck_df['0_latency']/1000 if n_bottleneck > 0 else None
    print(f"  Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    if n_lat is not None:
        print(f"  Normal p50: {n_lat.median():.1f}ms")
    if b_lat is not None:
        print(f"  BN p50: {b_lat.median():.1f}ms")
    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4),
        'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  Saved: {results_path}")
    os.remove(local_csv)
    print(f"  Local file deleted.")
    print(f"\n{'='*50}")
    print(f"⚠ {FILE_ID} — {status}")
    print(f"{'='*50}")

else:
    n_lat = normal_df['0_latency'] / 1000
    b_lat = bottleneck_df['0_latency'] / 1000

    rpc_rates   = df[rpc_cols].mean()
    flagged_set = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates>0].index)
    never_set   = set(c.replace('_label_RPC','')
                      for c in rpc_rates[rpc_rates==0].index)
    df['n_bn_rpcs'] = df[rpc_cols].sum(axis=1)
    multi = df[df['label_trace']==1]['n_bn_rpcs']

    res_ratios = {
        'cpu':    bottleneck_df[cpu_cols].mean().mean() /
                  normal_df[cpu_cols].mean().mean(),
        'memory': bottleneck_df[mem_cols].mean().mean() /
                  normal_df[mem_cols].mean().mean(),
        'net_rx': bottleneck_df[net_rx_cols].mean().mean() /
                  normal_df[net_rx_cols].mean().mean(),
        'net_tx': bottleneck_df[net_tx_cols].mean().mean() /
                  normal_df[net_tx_cols].mean().mean(),
    }

    print(f"\n[L2] Flagged nodes ({len(flagged_set)}): "
          f"{sorted(flagged_set, key=int)}")
    print(f"     Never flagged ({len(never_set)}): "
          f"{sorted(never_set, key=int)}")
    print(f"     Simultaneous RPCs: always {int(multi.mode()[0])} "
          f"| Rigid: {multi.nunique()==1}")
    print(f"     Entry p50 — Normal: {n_lat.median():.1f}ms  "
          f"BN: {b_lat.median():.1f}ms  "
          f"Ratio: {b_lat.median()/n_lat.median():.2f}x")
    print(f"     Resources: CPU={res_ratios['cpu']:.4f}x  "
          f"Mem={res_ratios['memory']:.4f}x  "
          f"NetRX={res_ratios['net_rx']:.4f}x")

    node_ratios = {}
    for col in latency_cols:
        node = col.replace('_latency','')
        nv   = normal_df[col].mean() / 1000
        bv   = bottleneck_df[col].mean() / 1000
        node_ratios[node] = {
            'normal_ms': round(nv,3), 'bn_ms': round(bv,3),
            'abs_diff':  round(bv-nv,3),
            'ratio':     round(bv/nv,4) if nv>0 else 0,
            'flagged':   node in flagged_set
        }

    lat_df = df[latency_cols].copy()
    lat_df.columns = [c.replace('_latency','') for c in latency_cols]
    corr   = lat_df.corr()
    fl     = [c for c in lat_df.columns if c in flagged_set]
    unfl   = [c for c in lat_df.columns if c not in flagged_set]
    ff     = corr.loc[fl, fl].values.copy() if len(fl)>1 else np.array([[np.nan]])
    np.fill_diagonal(ff, np.nan)
    fu     = corr.loc[fl, unfl].values if (len(fl)>0 and len(unfl)>0) else np.array([[0]])

    X_lat = df[latency_cols].values
    X_res = df[cpu_cols + mem_cols + net_rx_cols + net_tx_cols].values

    feature_aucs = {}
    for name, X in [('Latency only', X_lat),
                    ('Resource only', X_res)]:
        sc = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            X, y, cv=5, scoring='roc_auc')
        feature_aucs[name] = {'mean': round(sc.mean(),4),
                              'std':  round(sc.std(),4)}

    print(f"\n[L3] Latency-only AUC: "
          f"{feature_aucs['Latency only']['mean']:.4f}  "
          f"Resource-only AUC: "
          f"{feature_aucs['Resource only']['mean']:.4f}")
    print(f"     FF corr: {np.nanmean(ff):.4f}  "
          f"FU corr: {np.nanmean(fu):.4f}")

    scaler   = StandardScaler()
    X_lat_sc = scaler.fit_transform(X_lat)
    models   = {
        'Logistic Regression': (LogisticRegression(max_iter=1000,
                                 random_state=42), X_lat_sc),
        'Random Forest':       (RandomForestClassifier(n_estimators=200,
                                 random_state=42, n_jobs=-1), X_lat),
        'Gradient Boosting':   (GradientBoostingClassifier(n_estimators=100,
                                 random_state=42), X_lat),  # fixed
    }
    model_results = {}
    for name, (model, X) in models.items():
        auc_s = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
        f1_s  = cross_val_score(model, X, y, cv=5, scoring='f1')
        model_results[name] = {'auc': round(auc_s.mean(),4),
                               'f1':  round(f1_s.mean(),4),
                               'std': round(auc_s.std(),4)}

    baseline_auc = cross_val_score(
        RandomForestClassifier(n_estimators=100,
                               random_state=42, n_jobs=-1),
        X_lat, y, cv=5, scoring='roc_auc').mean()

    ablation = {}
    for i, col in enumerate(latency_cols):
        node = col.replace('_latency','')
        sc   = cross_val_score(
            RandomForestClassifier(n_estimators=100,
                                   random_state=42, n_jobs=-1),
            np.delete(X_lat, i, axis=1), y, cv=3,
            scoring='roc_auc').mean()
        ablation[node] = round(sc,4)

    drops   = {n: round(baseline_auc-sc,4) for n,sc in ablation.items()}
    drops_s = sorted(drops.items(), key=lambda x: x[1], reverse=True)

    print(f"\n[L4] RF AUC: {model_results['Random Forest']['auc']:.4f}  "
          f"F1: {model_results['Random Forest']['f1']:.4f}")
    print(f"     LR AUC: "
          f"{model_results['Logistic Regression']['auc']:.4f}  "
          f"GB AUC: {model_results['Gradient Boosting']['auc']:.4f}")
    print(f"     Top ablation: {drops_s[:3]}")

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Analysis: {FILE_NAME}\n'
                 f'{WORKFLOW} | {BN_TYPE} | {LOAD_RPS} RPS | Run {RUN_NUM}',
                 fontsize=12, fontweight='bold')

    ax = axes[0,0]
    ax.bar(['Normal','Bottleneck'], [n_normal, n_bottleneck],
           color=['#2196F3','#F44336'], edgecolor='white', width=0.5)
    ax.set_title('Class balance', fontweight='bold')
    ax.set_ylim(0, max(n_normal, n_bottleneck)*1.25)
    for i, val in enumerate([n_normal, n_bottleneck]):
        ax.text(i, val + max(n_normal,n_bottleneck)*0.02,
                f'{val:,}\n({val/n_total*100:.1f}%)',
                ha='center', fontsize=9, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[0,1]
    ax.bar(range(30), [1]*30,
           color=['#F44336' if str(n) in flagged_set
                  else '#E0E0E0' for n in range(30)],
           edgecolor='white', width=0.8)
    ax.set_xticks(range(30))
    ax.set_xticklabels([str(n) for n in range(30)], fontsize=6, rotation=90)
    ax.set_yticks([])
    ax.set_title(f'Flagged nodes ({len(flagged_set)}/30)', fontweight='bold')
    ax.spines[['top','right','left']].set_visible(False)

    ax = axes[0,2]
    nodes_s  = sorted(node_ratios.keys(), key=int)
    ratios_v = [node_ratios[n]['ratio'] for n in nodes_s]
    ax.bar(range(30), ratios_v,
           color=['#F44336' if node_ratios[n]['flagged']
                  else '#90CAF9' for n in nodes_s],
           edgecolor='white', width=0.7)
    ax.axhline(1.0, color='gray', linestyle='--', lw=1, alpha=0.7)
    ax.set_xticks(range(30))
    ax.set_xticklabels(nodes_s, fontsize=6, rotation=90)
    ax.set_title('Per-node latency ratio', fontweight='bold')
    ax.set_ylabel('BN / Normal')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,0]
    rv = list(res_ratios.values())
    ax.barh(list(res_ratios.keys()), rv,
            color=['#EF5350' if v>1.1 else '#FFA726'
                   if v>1.03 else '#66BB6A' for v in rv],
            edgecolor='white')
    ax.axvline(1.0, color='gray', linestyle='--', lw=1)
    ax.set_title('Resource ratios', fontweight='bold')
    for i, v in enumerate(rv):
        ax.text(v+0.001, i, f'{v:.4f}x', va='center', fontsize=9)
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,1]
    fa_vals = [feature_aucs[k]['mean'] for k in feature_aucs]
    ax.bar(range(2), fa_vals, color=['#4CAF50','#FF9800'],
           edgecolor='white', width=0.4)
    ax.set_xticks(range(2))
    ax.set_xticklabels(['Latency\nonly','Resource\nonly'], fontsize=9)
    ax.set_ylim(0.5, 1.05)
    ax.set_title('AUC by feature set (RF)', fontweight='bold')
    for i, v in enumerate(fa_vals):
        ax.text(i, v+0.008, f'{v:.4f}', ha='center',
                fontsize=10, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

    ax = axes[1,2]
    ax.barh(range(8), [v for _,v in drops_s[:8]][::-1],
            color=['#D32F2F' if v>0.008 else '#FF9800'
                   if v>0.004 else '#FFC107'
                   for _,v in drops_s[:8]][::-1],
            edgecolor='white')
    ax.set_yticks(range(8))
    ax.set_yticklabels([f'node {n}' for n,_ in drops_s[:8]][::-1], fontsize=8)
    ax.set_title(f'Node ablation (base={baseline_auc:.4f})', fontweight='bold')
    ax.set_xlabel('AUC drop')
    ax.spines[['top','right']].set_visible(False)

    plt.tight_layout()
    chart_path = os.path.join(CHARTS_DIR, f'{FILE_ID}_analysis.png')
    plt.savefig(chart_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Chart saved: {chart_path}")

    results = {
        'file_id': FILE_ID, 'file_name': FILE_NAME,
        'bn_type': BN_TYPE, 'workflow': WORKFLOW,
        'load_rps': LOAD_RPS, 'run_num': RUN_NUM,
        'n_total': n_total, 'n_normal': n_normal,
        'n_bottleneck': n_bottleneck,
        'bn_rate': round(bn_rate,4), 'status': status,
        'flagged_nodes': sorted(flagged_set, key=int),
        'never_flagged': sorted(never_set, key=int),
        'n_flagged': len(flagged_set),
        'always_same_subgraph': bool(multi.nunique()==1),
        'simultaneous_rpcs': int(multi.mode()[0]),
        'entry_latency': {
            'normal_p50_ms': round(n_lat.median(),2),
            'bn_p50_ms':     round(b_lat.median(),2),
            'ratio_p50':     round(b_lat.median()/n_lat.median(),4),
            'ratio_mean':    round(b_lat.mean()/n_lat.mean(),4),
        },
        'resource_ratios': {k: round(v,4) for k,v in res_ratios.items()},
        'feature_aucs':   feature_aucs,
        'model_results':  model_results,
        'baseline_auc':   round(baseline_auc,4),
        'ablation_top8':  {n: v for n,v in drops_s[:8]},
        'node_ratios':    node_ratios,
        'ff_corr': round(float(np.nanmean(ff)),4),
        'fu_corr': round(float(np.nanmean(fu)),4),
    }
    results_path = os.path.join(RESULTS_DIR, f'{FILE_ID}_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Results saved: {results_path}")
    os.remove(local_csv)
    print(f"Local file deleted.")
    print(f"\n{'='*50}")
    print(f"✓ {FILE_ID} COMPLETE")
    print(f"{'='*50}")

## Batch 7: Home Workflow — CPU bottleneck — 200 RPS (runs 0–6)
*Files: cpu_aug12_25min_200_[0-6]_graph_2.csv | 7 files | ~812,000 traces*
*Graph: graph_2 | 7 nodes (vs 30 in compose workflow)*

### The Home workflow has a completely different graph structure

The compose workflow had 30 nodes across a complex call chain.
The Home workflow has only **7 nodes** — a much simpler topology.
This is architecturally correct: reading the home timeline is a
simpler operation than composing a post.

### A new minimal pattern: only 2 nodes flagged — Pattern G

All 7 runs flag exactly the same 2 nodes: **{3, 4}**
This is the smallest bottleneck subgraph in the entire dataset.

| Property | Home Pattern G | Compare to smallest compose pattern |
|----------|---------------|-------------------------------------|
| Flagged nodes | {3, 4} | Pattern C: {0,1,2,22} — 4 nodes |
| Total nodes | 7 | 30 |
| % of graph flagged | 28.6% | 13.3% |
| Entry ratio | 0.96–0.98× | 1.31–9.35× |

### Entry latency is INVERTED — bottleneck traces are FASTER at entry

In all 7 runs the entry p50 ratio is **0.96–0.98×** — the entry point
(node 0) is slightly faster during bottleneck traces, not slower.
This is the same paradox as node 26 in compose — the constraint is
deep in the graph, and upstream nodes receive fewer/lighter requests
because the downstream bottleneck throttles the flow.

This completely inverts the detection approach: you cannot detect
this bottleneck by watching entry latency. The signal lives
exclusively in nodes 3 and 4.

### Detection performance: excellent despite minimal signal

| Metric | Range across 7 runs |
|--------|-------------------|
| RF AUC | 0.933–0.938 |
| GB AUC | 0.933–0.937 |
| LR AUC | **0.499–0.538** |
| F1 | 0.836–0.872 |

LR AUC of 0.499–0.538 is essentially random — the weakest LR
performance in the entire dataset. The decision boundary for
this 2-node pattern is completely non-linear. RF and GB handle
it easily because they can build node-specific thresholds.

### Node 4 is overwhelmingly dominant in ablation

In all 7 runs: node 4 drops 0.058–0.070 AUC when removed.
Node 3 drops only 0.006–0.010. Node 5 (unflagged) drops 0.005–0.008.

Node 4 alone carries almost all the discriminative information.
This is the strongest single-node dominance in the dataset —
even stronger than node 0 in Pattern C (July compose files).

### FF correlation is very high: 0.718–0.773
The 2 flagged nodes are tightly correlated (r=0.72–0.77).
They move together as a unit — when one slows, both slow.
This is consistent with them being sequential steps in a
tight sub-chain within the 7-node home timeline graph.

### Resource signals: CPU elevated but not reliable
CPU ratio: 1.03–1.57× (run 3 highest at 1.56×)
Memory ratio: 1.00–1.01× (essentially flat)
Resource-only AUC: 0.720–0.905 — variable, sometimes useful

Same conclusion as compose: resource signals may detect THAT a
bottleneck exists but not WHERE. Latency-only detection is
more consistent (0.933–0.937 vs 0.720–0.905).

### Scale note: largest files in the dataset
The home workflow files are massive — 49,000 to 128,000 traces each.
Total: ~812,000 traces in 7 files alone. The home timeline is
read far more often than posts are composed, hence the volume.

---
## ═══════════════════════════════════════════════════════
## FULL DATASET ANALYSIS COMPLETE — ALL 196 FILES
## ═══════════════════════════════════════════════════════

### Final file count
| Workflow | Files | Traces | Patterns found |
|---------|-------|--------|---------------|
| Compose | 188 files | ~2,500,000 | A, B, C, D, E, F |
| Home | 7 files | ~812,000 | G |
| **Total** | **195 files** | **~3,312,000** | **7 patterns** |

*Note: 1 file was a duplicate download (mem_sep25 run 9 counted twice)*

### Complete Pattern Taxonomy — Final Version

| Pattern | Workflow | Nodes | Flagged set | Bottleneck type |
|---------|---------|-------|-------------|----------------|
| A | Compose | 15 | {4,5,7,8,11,12,13,14,18,19,20,21,26,27,28} | CPU only |
| B | Compose | 9 | {4,5,13,14,20,21,26,27,28} | CPU only |
| C | Compose | 4 | {0,1,2,22} | CPU only |
| D | Compose | 11 | {0,1,2,13,14,20,21,22,26,27,28} | CPU+Memory |
| E | Compose | 19 | {0,1,2,4,5,7,8,11,12,13,14,18,19,20,21,22,26,27,28} | CPU+Memory (rare) |
| F | Compose | 11 | {0,1,2,4,5,7,8,11,12,18,19} | Network only |
| **G** | **Home** | **2** | **{3,4}** | **CPU only** |

### The universal conclusion across all workflows and all stress types

**Constraints are architectural, not resource-specific.**

Whether the stress is CPU, memory, combined, or network —
whether the workflow is compose (30 nodes) or home (7 nodes) —
the bottleneck always follows a structurally determined path
through the service call graph. The same services are always
the constraint for a given injection location, regardless of
what resource is being stressed.

This is Theory of Constraints applied to distributed systems:
the constraint is a property of the system architecture,
not a property of the workload.

---
## Next step: Paper writing phase
*All analysis complete. Ready to structure the academic paper.*